# DATASCAPE 2026 — E-Commerce Package Quality Control (EVA02-Large + Diagnosis Data)
## Klasifikasi kondisi paket: **Aman / Rusak / Lainnya** — metrik: Macro F1

### 🤖 AI Declaration
Notebook ini dibuat dengan bantuan AI (Claude, Anthropic) **hanya sebagai alat bantu** (vibe coding, brainstorming, debugging). AI/LLM **tidak** digunakan untuk melakukan inference/prediksi secara langsung maupun manual. Seluruh prediksi dihasilkan oleh model EVA02 yang dilatih di notebook ini pada data yang disediakan panitia.

---

### Mengapa ada versi ini
`eva001` mendapat **OOF Macro F1 ≈ 0.905 (0.911 setelah bias)** tetapi **skor public hanya 0.83**. Selisih ±0.07 berarti salah satu dari dua hal: validasi terlalu optimis, atau test berbeda dari train. Notebook diagnosis menunjukkan **keduanya terjadi**. Versi ini menggabungkan diagnosis ke dalam pipeline dan mengubah preprocessing / validasi sesuai temuannya.

| # | Temuan diagnosis | Akibat di eva001 | Aksi di notebook ini |
|---|---|---|---|
| 1 | 44% gambar punya kembaran (dHash ≤ 6, termasuk cermin); fold lama acak | Kembaran jatuh di fold train **dan** validasi → OOF 0.905 bocor/optimis | **StratifiedGroupKFold** berdasarkan kelompok kembaran (tidak ada kelompok yang terbelah) |
| 2 | Dedup lama membuang 1.327 gambar yang sebagian adalah versi lain (resolusi/format) dari foto yang sama | Model kehilangan contoh "foto sama, kualitas beda" — persis pergeseran train→test | **Tidak ada gambar dibuang**; sampler diberi bobot `ukuran_kelompok^-0.5`, jumlah sampel/epoch tetap (waktu latih sama) |
| 3 | 95 kelompok kembaran berlabel bertentangan (88 Aman-vs-Rusak) | Label noise di training | Pasangan kembaran (dHash ≤ 2) berlabel beda **dikeluarkan dari training** (tetap dipakai di validasi) |
| 4 | Adversarial validation AUC ≈ 0.87: test lebih kecil & lebih terkompresi (median 24,9 KB vs 53,1 KB; 224×224 = 17% test vs 2% train; WEBP/PNG lebih banyak) | Model dilatih pada kualitas "lebih bagus" daripada test | Augmentasi **Downscale + kompresi JPEG/WebP** menggantikan kompresi lama; **bobot shift (density-ratio)** untuk evaluasi OOF & optimasi bias |
| 5 | Metadata saja (tanpa gambar) sudah Macro F1 ≈ 0.79; Lainnya = 200×200 JPEG | Model bisa mencontek ukuran/blur, bukan isi | Degradasi diterapkan ke **semua kelas**; tabel prediksi per ukuran gambar test di akhir untuk memeriksa jalan pintas |
| 6 | 32–40% test punya kembaran di train; salin label kembaran akurat ±96% | Informasi gratis tidak dipakai (override lama hanya kena 17 gambar) | **Twin blend**: label kembaran ditambahkan sebagai petunjuk, bobotnya dipilih dari OOF yang jujur |

### Yang sengaja TIDAK diubah
Model (EVA02-L), resolusi 392, LR/epoch/EMA/mixup, TTA — supaya perubahan skor bisa dikaitkan ke perubahan **data & validasi**.

> ⚠️ **OOF versi ini akan lebih rendah daripada eva001. Itu normal** (fold sekarang tidak bocor). Bandingkan OOF versi ini dengan skor public, bukan dengan OOF eva001.

### Struktur
1 Konfigurasi → 2 Environment → 3 Data → 4 Cache → **5 Diagnosis (3 skrip)** → **6 Preprocessing dari diagnosis** → 7 Dataset/Augmentasi → 8 Training → 9 Jalankan → 10 OOF/test → 11 Evaluasi + bias + twin → 12 Submission → 13 Manifest.


## 1. Konfigurasi
Semua parameter di `Config`. Urutan `CLASSES` = `Aman, Rusak, Lainnya` (sama dengan label di submission).

In [1]:
import os, sys, gc, json, math, time, random, logging, warnings, shutil
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")

class Config:
    # ---------- Umum ----------
    SEED            = 42
    COMPETITION     = "DATASCAPE2026-PackageQC"
    CLASSES         = ["Aman", "Rusak", "Lainnya"]
    NUM_CLASSES     = 3

    # ---------- Data (None = deteksi otomatis di /kaggle/input) ----------
    INPUT_ROOT      = Path("/kaggle/input")
    TRAIN_DIR       = "/kaggle/input/competitions/datascape-2026-data-mining-competition/Train_Set"     
    TEST_DIR        = "/kaggle/input/competitions/datascape-2026-data-mining-competition/Test_Set"     
    SAMPLE_SUB      = "/kaggle/input/competitions/datascape-2026-data-mining-competition/sample_submission.csv"     
    IMG_EXTS        = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    CACHE_DIR       = Path("/tmp/img_cache")   # cache resize (tidak ikut jadi output)
    CACHE_MAX_SIDE  = 640

    # ---------- Output ----------
    OUT_ROOT        = Path("/kaggle/working/DATASCAPE2026_v2")
    MODELS_DIR      = OUT_ROOT / "models"
    PROBS_DIR       = OUT_ROOT / "probs"
    OUTPUTS_DIR     = OUT_ROOT / "outputs"
    LOGS_DIR        = OUT_ROOT / "logs"
    SUBMISSION_PATH = Path("/kaggle/working/submission.csv")

    # ---------- Lintas-sesi ----------
    # PENTING: JANGAN isi dengan output eva001 (fold-nya berbeda -> OOF/model tidak kompatibel).
    # Hanya untuk melanjutkan sesi dari notebook INI. Daftar folder output sesi sebelumnya, contoh:
    # [Path("/kaggle/input/datascape-eva-sesi1/DATASCAPE2026_outputs")]
    PREV_OUTPUT_DIRS = []

    # ---------- Cross-validation ----------
    N_FOLDS         = 5
    TRAIN_FOLDS     = [0, 1, 2, 3, 4]   # fold selesai otomatis dilewati; dihentikan rapi oleh TIME_BUDGET_H

    # ---------- Model (HANYA EVA02) ----------
    MODEL_NAME      = "eva02_large"
    TIMM_ID         = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"
    IMG_SIZE        = 392        # kelipatan 14 (28 patch). Opsi: 336 (lebih cepat) / 448 (native, paling akurat tapi lambat)
    BATCH_PER_GPU   = 8
    GRAD_CKPT       = True
    GRAD_ACCUM_TARGET = 32
    DROP_PATH       = 0.1

    # ---------- Training ----------
    EPOCHS          = 5
    WARMUP_EPOCHS   = 1
    LR              = 3e-5
    LAYER_DECAY     = 0.85
    WEIGHT_DECAY    = 0.05
    LABEL_SMOOTHING = 0.1
    GRAD_CLIP       = 1.0
    EARLY_STOP_PATIENCE = 2
    NUM_WORKERS     = 4
    RESUME          = True
    SAMPLER_POWER   = 0.5        # 0 = tanpa rebalancing, 1 = seimbang penuh
    LOSS_CW_POWER   = 0.0        # 0 = loss tanpa class-weight (hindari kompensasi ganda)
    MIXUP_ALPHA     = 0.2
    CUTMIX_ALPHA    = 1.0
    MIXCUT_P        = 0.3        # lebih rendah dari lama: CutMix bisa menimpa area kerusakan
    MIXUP_OFF_LAST_EPOCH = True

    # ---------- Anggaran waktu ----------
    TIME_BUDGET_H     = 11.3
    FIRST_EPOCH_EST_H = 1.0
    SAVE_EVERY_MIN    = 10

    # ---------- Augmentasi ----------
    RRC_SCALE       = (0.7, 1.0)
    DOWNSCALE_RANGE = (0.35, 0.85)   # perkecil lalu perbesar lagi (meniru test: lebih kecil/kabur, mis. 224x224)
    DOWNSCALE_P     = 0.30
    COMPRESS_Q      = (30, 90)       # kualitas JPEG/WebP (test: WEBP 19.5%, PNG 4.7%, byte median 2x lebih kecil)
    COMPRESS_P      = 0.40

    # ---------- Hasil diagnosis -> parameter aksi ----------
    GROUP_HAM        = 6      # dHash <= 6 (termasuk cermin) = satu kelompok; kelompok tidak boleh terbelah antar fold
    NOISE_HAM        = 2      # kembaran (dHash <= 2) berlabel beda = label noise -> keluar dari TRAINING (tetap ikut validasi)
    GROUP_W_POWER    = 0.5    # bobot sampler per gambar ∝ ukuran_kelompok^-0.5 (kelompok besar = duplikat/gambar polos)
    EPOCH_SAMPLES    = 8700   # sampel per epoch (tetap, setara eva001) -> waktu latih sama walau tidak ada gambar dibuang
    SHIFT_W_POWER    = 0.5    # bobot density-ratio train->test (adversarial), dipangkatkan ini agar tidak ekstrem
    SHIFT_W_CLIP     = (0.2, 5.0)
    TWIN_HAM         = 4      # gambar yang punya kembaran berlabel di train (dHash <= 4): labelnya jadi petunjuk tambahan
    USE_TWIN_BLEND   = True
    TWIN_W_GRID      = (0.5, 1.0, 1.5, 2.0, 3.0)   # bobot (dalam satuan logit) yang dicoba pada OOF
    BIAS_USE_SHIFT_W = True   # optimasi bias memakai Macro F1 berbobot-shift

    # ---------- Inferensi / pasca-proses ----------
    TTA             = True       # original + hflip
    OPTIMIZE_BIAS   = True       # bias logit per-kelas dari OOF utk memaksimalkan Macro F1
    BIAS_MIN_GAIN   = 0.001

CFG = Config()
SESSION_T0 = time.time()
def elapsed_h(): return (time.time() - SESSION_T0) / 3600.0

for d in [CFG.OUT_ROOT, CFG.MODELS_DIR, CFG.PROBS_DIR, CFG.OUTPUTS_DIR, CFG.LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger(CFG.COMPETITION)
logger.setLevel(logging.INFO); logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for h in [logging.StreamHandler(sys.stdout), logging.FileHandler(CFG.LOGS_DIR / "execution.log")]:
    h.setFormatter(_fmt); logger.addHandler(h)
log = logger.info

def all_roots():
    return [CFG.OUT_ROOT] + [Path(p) for p in CFG.PREV_OUTPUT_DIRS]

def find_in_roots(rel):
    """Cari file relatif (mis. 'models/eva02_large/fold0_best.pt') di output sesi ini lalu sesi-sesi sebelumnya."""
    for r in all_roots():
        p = Path(r) / rel
        if p.exists():
            return p
    return None

log(f"Output: {CFG.OUT_ROOT} | folds={CFG.TRAIN_FOLDS} | budget={CFG.TIME_BUDGET_H}h | img={CFG.IMG_SIZE}")

2026-10-02 09:50:51,832 | INFO | Output: /kaggle/working/DATASCAPE2026_v2 | folds=[0, 1, 2, 3, 4] | budget=11.3h | img=392


## 2. Environment Setup
T4 (CC 7.5) tidak punya bfloat16 hardware → AMP dipaksa **float16** + GradScaler. Dua GPU dipakai via `DataParallel`.

In [2]:
import subprocess
def pip_install(pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, capture_output=True, text=True)
    if r.returncode != 0:
        log(f"pip install warning: {r.stderr[-500:]}")

pip_install(["timm>=1.0.0", "albumentations>=2.0.0", "opencv-python-headless", "scikit-learn"])

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
import timm
from timm.data import Mixup
from timm.utils import ModelEmaV2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
ImageFile.LOAD_TRUNCATED_IMAGES = True

def seed_everything(seed=CFG.SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = max(1, torch.cuda.device_count())
if torch.cuda.is_available():
    cc_major, _ = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cc_major >= 8 else torch.float16
else:
    AMP_DTYPE = torch.float32
USE_FP16_SCALER = (AMP_DTYPE == torch.float16)

log(f"Device: {DEVICE} | GPUs: {N_GPUS} | AMP: {AMP_DTYPE} | timm {timm.__version__} | torch {torch.__version__} | albumentations {A.__version__}")
for i in range(torch.cuda.device_count()):
    log(f"  GPU{i}: {torch.cuda.get_device_name(i)}")

GLOBAL_BS = CFG.BATCH_PER_GPU * N_GPUS
ACCUM = max(1, CFG.GRAD_ACCUM_TARGET // GLOBAL_BS)
log(f"Batch global {GLOBAL_BS} x accum {ACCUM} = efektif {GLOBAL_BS * ACCUM}")

2026-10-02 09:51:14,419 | INFO | Device: cuda | GPUs: 2 | AMP: torch.float16 | timm 1.0.26 | torch 2.10.0+cu128 | albumentations 2.0.8
2026-10-02 09:51:14,420 | INFO |   GPU0: Tesla T4
2026-10-02 09:51:14,421 | INFO |   GPU1: Tesla T4
2026-10-02 09:51:14,422 | INFO | Batch global 16 x accum 2 = efektif 32


## 3. Deteksi Data & Metadata
Struktur yang diharapkan: `.../Train_Set/{Aman,Lainnya,Rusak}/*.jpg`, `.../Test_Set/*.jpg`, `sample_submission.csv`. Folder dideteksi otomatis; kalau gagal, isi `CFG.TRAIN_DIR / TEST_DIR / SAMPLE_SUB` manual.

In [3]:
def scan_images(root: Path):
    return sorted([p for p in Path(root).rglob("*") if p.suffix.lower() in CFG.IMG_EXTS])

def _find_class_root():
    want = {c.lower() for c in CFG.CLASSES}
    base_depth = str(CFG.INPUT_ROOT).count(os.sep)
    for root, dirs, _ in os.walk(CFG.INPUT_ROOT):
        if want <= {d.lower() for d in dirs}:
            return Path(root)
        if root.count(os.sep) - base_depth >= 5:
            dirs[:] = []
    return None

def _find_test_dir(train_dir):
    best, best_n = None, 0
    base_depth = str(CFG.INPUT_ROOT).count(os.sep)
    for root, dirs, _ in os.walk(CFG.INPUT_ROOT):
        for d in dirs:
            if d.lower().startswith("test"):
                p = Path(root) / d
                if train_dir in p.parents or p == train_dir:
                    continue
                n = len(scan_images(p))
                if n > best_n: best, best_n = p, n
        if root.count(os.sep) - base_depth >= 5:
            dirs[:] = []
    return best

def _find_sample_sub():
    for root, _, files in os.walk(CFG.INPUT_ROOT):
        for f in files:
            if f.lower() == "sample_submission.csv":
                return Path(root) / f
    return None

if CFG.TRAIN_DIR is None:
    CFG.TRAIN_DIR = _find_class_root()
assert CFG.TRAIN_DIR is not None, "Folder train (berisi Aman/Rusak/Lainnya) tidak ditemukan. Isi CFG.TRAIN_DIR manual."
if CFG.TEST_DIR is None:
    CFG.TEST_DIR = _find_test_dir(Path(CFG.TRAIN_DIR))
assert CFG.TEST_DIR is not None, "Folder test tidak ditemukan. Isi CFG.TEST_DIR manual."
if CFG.SAMPLE_SUB is None:
    CFG.SAMPLE_SUB = _find_sample_sub()
CFG.TRAIN_DIR, CFG.TEST_DIR = Path(CFG.TRAIN_DIR), Path(CFG.TEST_DIR)
log(f"TRAIN_DIR={CFG.TRAIN_DIR}\nTEST_DIR={CFG.TEST_DIR}\nSAMPLE_SUB={CFG.SAMPLE_SUB}")

LABEL2ID = {c: i for i, c in enumerate(CFG.CLASSES)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}
_lower2canon = {c.lower(): c for c in CFG.CLASSES}

rows = []
for d in sorted(CFG.TRAIN_DIR.iterdir()):
    if d.is_dir() and d.name.lower() in _lower2canon:
        lab = _lower2canon[d.name.lower()]
        for p in scan_images(d):
            rows.append(dict(path=str(p), rel=str(p.relative_to(CFG.TRAIN_DIR)), label=lab))
train_df = pd.DataFrame(rows)
train_df["target"] = train_df.label.map(LABEL2ID)

test_df = pd.DataFrame([dict(path=str(p), filename=p.name) for p in scan_images(CFG.TEST_DIR)])
test_df = test_df.sort_values("filename").reset_index(drop=True)
assert test_df.filename.is_unique, "Nama file test tidak unik."

cls_counts = train_df.label.value_counts()
log(f"Train: {len(train_df)} | Test: {len(test_df)} | Total: {len(train_df)+len(test_df)} (target 15.281)")
log(f"Distribusi kelas: {cls_counts.to_dict()} | imbalance {cls_counts.max()/cls_counts.min():.2f}:1")

2026-10-02 09:51:14,476 | INFO | TRAIN_DIR=/kaggle/input/competitions/datascape-2026-data-mining-competition/Train_Set
TEST_DIR=/kaggle/input/competitions/datascape-2026-data-mining-competition/Test_Set
SAMPLE_SUB=/kaggle/input/competitions/datascape-2026-data-mining-competition/sample_submission.csv
2026-10-02 09:51:28,659 | INFO | Train: 12224 | Test: 3057 | Total: 15281 (target 15.281)
2026-10-02 09:51:28,660 | INFO | Distribusi kelas: {'Aman': 8800, 'Rusak': 3001, 'Lainnya': 423} | imbalance 20.80:1


## 4. Cache Resize + Validasi File Rusak
Foto lapangan sering beresolusi besar → decode di 4 core CPU jadi bottleneck GPU. Semua gambar di-resize **sekali** (sisi terpanjang ≤ 640 px, JPEG q95) ke `/tmp`; sekaligus mendeteksi file corrupt. Cache tidak ikut ke output notebook.

In [4]:
def build_cache(df, split):
    cdir = CFG.CACHE_DIR / split; cdir.mkdir(parents=True, exist_ok=True)
    def work(item):
        i, p = item
        dst = cdir / f"{i:06d}.jpg"
        if dst.exists() and dst.stat().st_size > 0:
            return i, str(dst)
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None:
            try:
                with Image.open(p) as im:
                    img = cv2.cvtColor(np.array(im.convert("RGB")), cv2.COLOR_RGB2BGR)
            except Exception:
                return i, None
        h, w = img.shape[:2]; m = max(h, w)
        if m > CFG.CACHE_MAX_SIDE:
            s = CFG.CACHE_MAX_SIDE / m
            img = cv2.resize(img, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        return i, str(dst)
    out = [None] * len(df)
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, pth in tqdm(ex.map(work, enumerate(df.path.tolist())), total=len(df), desc=f"cache {split}"):
            out[i] = pth
    return out

t0 = time.time()
train_df["fpath"] = build_cache(train_df, "train")
test_df["fpath"]  = build_cache(test_df, "test")
bad_tr = train_df[train_df.fpath.isna()]
bad_te = test_df[test_df.fpath.isna()]
for p in bad_tr.path: log(f"[CORRUPT-TRAIN-SKIP] {p}")
for p in bad_te.path: log(f"[CORRUPT-TEST] {p} -> akan diprediksi dgn prior kelas terbanyak")
train_df = train_df[train_df.fpath.notna()].reset_index(drop=True)
log(f"Cache selesai {time.time()-t0:.0f}s | corrupt train={len(bad_tr)}, test={len(bad_te)} | train valid={len(train_df)}")

cache train:   0%|          | 0/12224 [00:00<?, ?it/s]

cache test:   0%|          | 0/3057 [00:00<?, ?it/s]

libpng warning: eXIf: duplicate


2026-10-02 09:52:37,356 | INFO | Cache selesai 69s | corrupt train=0, test=0 | train valid=12224


## 5. Diagnosis Data (dari `data-diagnosis.ipynb`)
Tiga skrip diagnosis dijalankan **di dalam pipeline** (CPU, ±5 menit; tidak melatih model). Hasilnya (`RESULT`, `FOLLOWUP`) dipakai langsung oleh bagian 6.
Baca output + contact sheet (`DATASCAPE2026_v2/diagnosis/sheet_*.png`) — terutama `sheet_label_conflicts.png` dan `sheet_Lainnya.png`.

**5.1** manifest metadata, jalan pintas, adversarial validation, kembaran, fold berkelompok

In [5]:
# =============================================================================
# DATASCAPE 2026 - DIAGNOSIS HARI PERTAMA (CPU saja, tanpa melatih model)
#
# Tujuan: menjawab pertanyaan tentang DATA sebelum memilih model:
#   1. Kelas dikumpulkan dari sumber berbeda? (jalan pintas metadata/statistik)
#   2. Train dan test berbeda distribusi? (adversarial validation)
#   3. Ada kembaran / foto yang sama dengan label berbeda? (kebocoran + derau label)
#   4. Seperti apa isi tiap kelas dan batas Aman/Rusak/Lainnya? (contact sheet)
# Hasil akhirnya juga membuat fold berkelompok (fold_assignments_grouped.csv).
#
# Pakai di Kaggle: tempel semua isi file ke satu sel, lalu jalankan.
# Atur TRAIN_DIR / TEST_DIR manual bila deteksi otomatis gagal.
# Tidak butuh pustaka tambahan di luar numpy, pandas, PIL, scikit-learn.
# =============================================================================
import os
import sys
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFile, ImageOps
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold,
                                     cross_val_predict)

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

CLASSES = ["Aman", "Rusak", "Lainnya"]          # urutan indeks target 0,1,2
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".jfif"}
SEED = 42
HAMMING_THR = CFG.GROUP_HAM

# Isi manual bila perlu, mis. Path("/kaggle/input/NAMA-DATASET/Train")
TRAIN_DIR = str(CFG.TRAIN_DIR)
TEST_DIR = str(CFG.TEST_DIR)
OUT_DIR = CFG.OUT_ROOT / "diagnosis"


# ----------------------------------------------------------------------------- data
def find_dirs(base="/kaggle/input", max_depth=4):
    """Cari folder Train (berisi subfolder Aman/Rusak/Lainnya) dan folder Test."""
    want = {c.lower() for c in CLASSES}
    base = Path(base)
    train = None
    for root, dirs, _ in os.walk(base):
        depth = len(Path(root).relative_to(base).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        if want <= {d.lower() for d in dirs}:
            train = Path(root)
            break
    test = None
    if train is not None:
        for sib in sorted(train.parent.iterdir()):
            if sib.is_dir() and "test" in sib.name.lower():
                test = sib
                break
    return train, test


def scan(root):
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXTS)


def build_frames(train_dir, test_dir):
    rows = []
    for sub in Path(train_dir).iterdir():
        if not sub.is_dir():
            continue
        match = [c for c in CLASSES if c.lower() == sub.name.lower()]
        if not match:
            continue
        for p in scan(sub):
            rows.append(dict(path=str(p), filename=p.name, label=match[0]))
    tr = pd.DataFrame(rows)
    tr["target"] = tr.label.map({c: i for i, c in enumerate(CLASSES)})
    te = pd.DataFrame([dict(path=str(p), filename=p.name) for p in scan(test_dir)])
    return tr.reset_index(drop=True), te.reset_index(drop=True)


# ----------------------------------------------------------------------------- probe
_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def popcount64(x):
    out = np.zeros(x.shape, dtype=np.uint8)
    v = x.copy()
    for _ in range(8):
        out += _POP[(v & np.uint64(0xFF)).astype(np.uint8)]
        v >>= np.uint64(8)
    return out


def dhash_from_L(img_l):
    g = np.asarray(img_l.resize((9, 8), Image.LANCZOS), dtype=np.int16)
    bits = (g[:, :-1] < g[:, 1:]).flatten()
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return v


def flat_region_sigma(gray, block=16, flat_frac=0.08):
    """Estimasi derau dari 8% blok paling datar (laplacian), skala 0-255."""
    h, w = gray.shape
    hh, ww = (h // block) * block, (w // block) * block
    if hh < block * 4 or ww < block * 4:
        return np.nan
    blocks = (gray[:hh, :ww].reshape(hh // block, block, ww // block, block)
              .transpose(0, 2, 1, 3).reshape(-1, block, block))
    keep = max(4, int(len(blocks) * flat_frac))
    sel = blocks[np.argsort(blocks.reshape(len(blocks), -1).std(1))[:keep]]
    lap = (sel[:, :-2, :-2] - 2 * sel[:, 1:-1, :-2] + sel[:, 2:, :-2]
           - 2 * sel[:, :-2, 1:-1] + 4 * sel[:, 1:-1, 1:-1] - 2 * sel[:, 2:, 1:-1]
           + sel[:, :-2, 2:] - 2 * sel[:, 1:-1, 2:] + sel[:, 2:, 2:])
    return float(np.sqrt(np.pi / 2) * np.abs(lap).mean() / 6.0)


def probe_image(path):
    rec = dict(w=0, h=0, mode="", fmt="", nbytes=0, exif=0, dh=0, dhm=0,
               bright=np.nan, contrast=np.nan, sat=np.nan, noise=np.nan,
               blur=np.nan, ok=0)
    try:
        rec["nbytes"] = os.path.getsize(path)
        with Image.open(path) as im:
            rec["w"], rec["h"] = im.size
            rec["mode"], rec["fmt"] = im.mode, im.format or ""
            try:
                rec["exif"] = int(len(im.getexif()) > 0)
            except Exception:
                pass
            im = ImageOps.exif_transpose(im)
            if im.mode in ("RGBA", "LA", "P"):
                im = im.convert("RGBA")
                im = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im)
            rgb = im.convert("RGB")
        gray_full = np.asarray(rgb.convert("L"), dtype=np.float32)
        H, W = gray_full.shape
        s = min(256, H, W)
        y0, x0 = (H - s) // 2, (W - s) // 2
        rec["noise"] = flat_region_sigma(gray_full[y0:y0 + s, x0:x0 + s])   # resolusi asli

        small = rgb.resize((256, 256), Image.BILINEAR)
        a = np.asarray(small, dtype=np.float32) / 255.0
        g = a @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
        mx, mn = a.max(-1), a.min(-1)
        rec["bright"] = float(g.mean())
        rec["contrast"] = float(g.std())
        rec["sat"] = float(np.where(mx > 1e-6, (mx - mn) / (mx + 1e-6), 0).mean())
        lap = (g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:] - 4 * g[1:-1, 1:-1])
        rec["blur"] = float(lap.var())
        gl = small.convert("L")
        rec["dh"] = dhash_from_L(gl)
        rec["dhm"] = dhash_from_L(ImageOps.mirror(gl))
        rec["ok"] = 1
    except Exception:
        pass
    return rec


def build_manifest(df, n_workers):
    with ThreadPoolExecutor(n_workers) as ex:
        recs = list(ex.map(probe_image, df.path.tolist()))
    m = pd.concat([df.reset_index(drop=True), pd.DataFrame(recs)], axis=1)
    m["ar"] = m.w / m.h.clip(lower=1)
    m["mp"] = m.w * m.h
    m["bpp"] = m.nbytes / m.mp.clip(lower=1)
    return m


# ----------------------------------------------------------------------------- groups
def hamming_pairs(dh, dhm, valid, thr, chunk=512):
    """Pasangan (i<j) dengan jarak dHash <= thr, termasuk versi cermin horizontal."""
    n = len(dh)
    pairs = []
    for i0 in range(0, n, chunk):
        a, am = dh[i0:i0 + chunk, None], dhm[i0:i0 + chunk, None]
        d = np.minimum(popcount64(a ^ dh[None, :]), popcount64(am ^ dh[None, :]))
        ii, jj = np.where(d <= thr)
        for p, q in zip(ii, jj):
            i, j = i0 + int(p), int(q)
            if i < j and valid[i] and valid[j]:
                pairs.append((i, j))
    return pairs


def union_groups(n, pairs):
    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, j in pairs:
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)
    return np.array([find(i) for i in range(n)])


# ----------------------------------------------------------------------------- audits
META_COLS = ["w", "h", "ar", "mp", "nbytes", "bpp", "exif", "is_png", "is_jpeg", "is_rgba"]
CONTENT_COLS = ["bright", "contrast", "sat", "noise", "blur"]


def add_flags(m):
    m = m.copy()
    m["is_png"] = (m.fmt == "PNG").astype(int)
    m["is_jpeg"] = (m.fmt == "JPEG").astype(int)
    m["is_rgba"] = (m["mode"] == "RGBA").astype(int)
    return m


def majority_macro_f1(shares):
    m = max(shares)
    return (2 * m / (1 + m)) / len(shares)


def audit_shortcuts(tr, groups):
    y = tr.target.to_numpy()
    shares = pd.Series(y).value_counts(normalize=True).tolist()
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
    out = {}
    for name, cols in (("metadata", META_COLS), ("metadata + statistik isi", META_COLS + CONTENT_COLS)):
        X = tr[cols].fillna(tr[cols].median())
        pred = cross_val_predict(HistGradientBoostingClassifier(max_iter=120, random_state=SEED),
                                 X, y, cv=cv, groups=groups)
        out[name] = dict(acc=float((pred == y).mean()), macro_f1=float(f1_score(y, pred, average="macro")))
    print("\n[1] JALAN PINTAS: label ditebak TANPA model gambar (CV berkelompok)")
    print(f"    patokan tebak-mayoritas: akurasi {max(shares):.4f}, Macro F1 {majority_macro_f1(shares):.4f}")
    for k, v in out.items():
        print(f"    {k:<26} akurasi {v['acc']:.4f} | Macro F1 {v['macro_f1']:.4f}")
    print("    Format berkas per kelas:")
    print(pd.crosstab(tr.label, tr.fmt).to_string().replace("\n", "\n      "))
    print("    Median per kelas (ukuran, byte, derau, ketajaman, kecerahan):")
    print(tr.groupby("label")[["w", "h", "nbytes", "noise", "blur", "bright"]].median().round(2)
          .to_string().replace("\n", "\n      "))
    return out


def adversarial_validation(tr, te):
    cols = META_COLS + CONTENT_COLS
    both = pd.concat([tr[cols], te[cols]], ignore_index=True)
    both = both.fillna(both.median())
    y = np.r_[np.zeros(len(tr)), np.ones(len(te))]
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=120, random_state=SEED),
                          both, y, cv=cv, method="predict_proba")[:, 1]
    auc = roc_auc_score(y, p)
    uni = {c: abs(roc_auc_score(y, both[c]) - 0.5) + 0.5 for c in cols if both[c].nunique() > 1}
    uni = pd.Series(uni).sort_values(ascending=False)
    print("\n[2] ADVERSARIAL VALIDATION: bisakah model membedakan train vs test?")
    print(f"    AUC = {auc:.4f}  (0.50 = tak bisa dibedakan, mendekati 1.0 = distribusi sangat beda)")
    print("    Fitur yang paling membedakan (AUC univariat):")
    print(uni.head(5).round(3).to_string().replace("\n", "\n      "))
    return auc, uni


def audit_duplicates(tr, te, groups_all):
    n_tr = len(tr)
    g_tr, g_te = groups_all[:n_tr], groups_all[n_tr:]
    tr = tr.assign(group=g_tr)
    sizes = pd.Series(groups_all).value_counts()
    multi = int(sizes[sizes > 1].sum())
    span = tr.groupby("group").agg(n=("label", "size"), nl=("label", "nunique"),
                                   labs=("label", lambda s: tuple(sorted(set(s)))))
    span = span[span.n > 1]
    conflict = span[span.nl > 1]
    train_groups = set(g_tr)
    leak = int(np.isin(g_te, list(train_groups)).sum())
    print("\n[3] KEMBARAN DAN DERAU LABEL (dHash <= %d, termasuk cermin)" % HAMMING_THR)
    print(f"    gambar dalam kelompok multi-gambar : {multi:,} dari {len(groups_all):,} "
          f"({multi / len(groups_all) * 100:.1f}%) | kelompok terbesar {int(sizes.max())}")
    print(f"    kelompok train multi-gambar        : {len(span):,}")
    print(f"    ... dengan LABEL BERTENTANGAN      : {len(conflict):,}")
    if len(conflict):
        print("    pasangan label yang bertentangan:",
              dict(Counter(conflict.labs.map(lambda t: " vs ".join(t)))))
    print(f"    citra TEST yang berbagi kelompok dgn train: {leak}/{len(te)} ({leak / max(len(te), 1) * 100:.1f}%)")
    print("    CATATAN: kelompok terbesar yang sangat besar sering = gambar polos/hampir kosong "
          "yang kebetulan berhash sama; periksa manual.")
    return tr, conflict


def fold_split(tr, n_folds=5):
    sgkf = StratifiedGroupKFold(n_folds, shuffle=True, random_state=SEED)
    tr = tr.copy()
    tr["fold"] = -1
    for f, (_, v) in enumerate(sgkf.split(tr, tr.target, tr.group)):
        tr.loc[tr.index[v], "fold"] = f
    for f in range(n_folds):
        a = set(tr.loc[tr.fold != f, "group"])
        b = set(tr.loc[tr.fold == f, "group"])
        assert not (a & b), f"kelompok bocor di fold {f}"
    print("\n[4] FOLD BERKELOMPOK (tidak ada kelompok yang terbelah)")
    print(pd.crosstab(tr.fold, tr.label).to_string().replace("\n", "\n    "))
    return tr


# ----------------------------------------------------------------------------- sheets
def contact_sheet(paths, title, cols=8, rows=4, cell=160):
    sheet = Image.new("RGB", (cols * cell, rows * cell + 22), (255, 255, 255))
    ImageDraw.Draw(sheet).text((6, 5), title, fill=(0, 0, 0))
    for k, p in enumerate(paths[:cols * rows]):
        try:
            with Image.open(p) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
                im = ImageOps.fit(im, (cell, cell))
            sheet.paste(im, ((k % cols) * cell, 22 + (k // cols) * cell))
        except Exception:
            pass
    return sheet


def conflict_sheet(tr, conflict, max_pairs=12, cell=200):
    rows = []
    for gid in list(conflict.index)[:max_pairs]:
        sub = tr[tr.group == gid]
        pick = []
        for lab in sub.label.unique()[:2]:
            pick.append(sub[sub.label == lab].iloc[0])
        rows.append(pick)
    if not rows:
        return None
    sheet = Image.new("RGB", (2 * cell, len(rows) * (cell + 18)), (255, 255, 255))
    d = ImageDraw.Draw(sheet)
    for r, pair in enumerate(rows):
        for c, rec in enumerate(pair):
            try:
                with Image.open(rec.path) as im:
                    im = ImageOps.fit(ImageOps.exif_transpose(im).convert("RGB"), (cell, cell))
                sheet.paste(im, (c * cell, r * (cell + 18) + 18))
            except Exception:
                pass
            d.text((c * cell + 4, r * (cell + 18) + 3), f"label: {rec.label}", fill=(200, 0, 0))
    return sheet


# ----------------------------------------------------------------------------- main
def run_diagnosis(train_dir=None, test_dir=None, out_dir=None, n_workers=4):
    t0 = time.time()
    train_dir, test_dir = train_dir or TRAIN_DIR, test_dir or TEST_DIR
    if train_dir is None or test_dir is None:
        a, b = find_dirs()
        train_dir, test_dir = train_dir or a, test_dir or b
    assert train_dir is not None and test_dir is not None, \
        "Folder Train/Test tidak ditemukan. Isi TRAIN_DIR dan TEST_DIR manual."
    out = Path(out_dir or OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    print(f"Train: {train_dir}\nTest : {test_dir}")

    tr, te = build_frames(train_dir, test_dir)
    print(f"Gambar train: {len(tr):,} | test: {len(te):,}")
    print("Distribusi kelas:", tr.label.value_counts().to_dict(),
          f"| rasio mayoritas/minoritas {tr.label.value_counts().max() / tr.label.value_counts().min():.2f}x")

    tr, te = add_flags(build_manifest(tr, n_workers)), add_flags(build_manifest(te, n_workers))
    print(f"Tidak terbaca: train {int((tr.ok == 0).sum())}, test {int((te.ok == 0).sum())} "
          f"| manifest selesai dalam {(time.time() - t0) / 60:.1f} menit")

    # kelompok kembaran (train + test bersama)
    allm = pd.concat([tr[["dh", "dhm", "ok"]], te[["dh", "dhm", "ok"]]], ignore_index=True)
    dh, dhm = allm.dh.to_numpy(dtype=np.uint64), allm.dhm.to_numpy(dtype=np.uint64)
    valid = allm.ok.to_numpy().astype(bool)
    pairs = hamming_pairs(dh, dhm, valid, HAMMING_THR)
    groups_all = union_groups(len(allm), pairs)
    tr["group"] = groups_all[:len(tr)]

    res = {}
    res["shortcuts"] = audit_shortcuts(tr, tr.group.to_numpy())
    res["adv_auc"], res["adv_features"] = adversarial_validation(tr, te)
    tr, conflict = audit_duplicates(tr, te, groups_all)
    tr = fold_split(tr)

    tr.drop(columns=["dh", "dhm"]).to_csv(out / "train_manifest.csv", index=False)
    te.drop(columns=["dh", "dhm"]).to_csv(out / "test_manifest.csv", index=False)
    tr[["filename", "path", "label", "target", "group", "fold"]].to_csv(
        out / "fold_assignments_grouped.csv", index=False)

    rng = np.random.default_rng(SEED)
    for lab in CLASSES:
        paths = tr[tr.label == lab].path.tolist()
        pick = [paths[i] for i in rng.permutation(len(paths))[:32]]
        contact_sheet(pick, f"{lab} (32 contoh acak)").save(out / f"sheet_{lab}.png")
    te_pick = [te.path[i] for i in rng.permutation(len(te))[:32]]
    contact_sheet(te_pick, "TEST (32 contoh acak, tanpa label)").save(out / "sheet_test.png")
    cs = conflict_sheet(tr, conflict)
    if cs is not None:
        cs.save(out / "sheet_label_conflicts.png")
    print(f"\nSelesai dalam {(time.time() - t0) / 60:.1f} menit. File ada di: {out}")
    print("WAJIB DILAKUKAN MANUAL: buka sheet_*.png dan baca gambarnya sendiri "
          "(terutama sheet_label_conflicts.png dan sheet_Lainnya.png).")
    res["train"], res["test"], res["conflict"] = tr, te, conflict
    return res


if __name__ == "__main__":
    RESULT = run_diagnosis()

Train: /kaggle/input/competitions/datascape-2026-data-mining-competition/Train_Set
Test : /kaggle/input/competitions/datascape-2026-data-mining-competition/Test_Set
Gambar train: 12,224 | test: 3,057
Distribusi kelas: {'Aman': 8800, 'Rusak': 3001, 'Lainnya': 423} | rasio mayoritas/minoritas 20.80x
Tidak terbaca: train 0, test 0 | manifest selesai dalam 2.3 menit

[1] JALAN PINTAS: label ditebak TANPA model gambar (CV berkelompok)
    patokan tebak-mayoritas: akurasi 0.7199, Macro F1 0.2790
    metadata                   akurasi 0.8160 | Macro F1 0.7876
    metadata + statistik isi   akurasi 0.8277 | Macro F1 0.7957
    Format berkas per kelas:
fmt      AVIF  JPEG  PNG  WEBP
      label                         
      Aman        1  7915   91   793
      Lainnya     0   423    0     0
      Rusak       0  2419    4   578
    Median per kelas (ukuran, byte, derau, ketajaman, kecerahan):
             w      h   nbytes  noise  blur  bright
      label                                        

**5.2** sensitivitas ambang kembaran, keandalan salin label, adversarial validation berkelompok, ukuran gambar

In [6]:
# =============================================================================
# DATASCAPE 2026 - DIAGNOSIS LANJUTAN (jalankan SETELAH datascape_diagnosis.py,
# di notebook yang sama, selama variabel RESULT masih ada)
#
# Menjawab pertanyaan yang muncul dari hasil pertama:
#   [A] Seberapa nyata "42% test punya kembaran di train"? (sensitivitas ambang)
#   [B] Seberapa bisa dipercaya menyalin label kembaran? (keandalan per kelas)
#   [C] Adversarial validation yang BENAR (CV berkelompok) + sumber pergeserannya
#   [D] Ukuran gambar per kelas vs test (tanda sumber data berbeda per kelas)
#   [E] Metadata saja: kelas mana yang sebenarnya bocor? (precision/recall)
#   [F] Contact sheet pasangan test-train untuk diverifikasi dengan mata
# =============================================================================
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict

try:                                   # dipakai bila skrip pertama sudah ditempel di notebook
    run_diagnosis
except NameError:                      # atau disimpan sebagai modul
    from datascape_diagnosis import *  # noqa: F401,F403
    from datascape_diagnosis import (CLASSES, META_COLS, CONTENT_COLS, SEED, HAMMING_THR,
                                     hamming_pairs, union_groups, popcount64, probe_image, Path)


def ensure_hashes(df, n_workers=4):
    if "dh" in df.columns and "dhm" in df.columns:
        return df
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(n_workers) as ex:
        recs = list(ex.map(probe_image, df.path.tolist()))
    df = df.copy()
    df["dh"] = [r["dh"] for r in recs]
    df["dhm"] = [r["dhm"] for r in recs]
    return df


def min_hamming_idx(q, qm, ref, exclude_self=False, chunk=512):
    """Untuk tiap query (asli dan cermin): jarak Hamming terkecil ke `ref` dan indeks tetangganya."""
    dmin = np.empty(len(q), dtype=np.int16)
    jmin = np.empty(len(q), dtype=np.int64)
    for i0 in range(0, len(q), chunk):
        a, am = q[i0:i0 + chunk, None], qm[i0:i0 + chunk, None]
        d = np.minimum(popcount64(a ^ ref[None, :]), popcount64(am ^ ref[None, :])).astype(np.int16)
        if exclude_self:
            r = np.arange(d.shape[0])
            d[r, i0 + r] = 99
        jmin[i0:i0 + chunk] = d.argmin(1)
        dmin[i0:i0 + chunk] = d.min(1)
    return dmin, jmin


# --------------------------------------------------------------------------- [A][B]
def twin_sensitivity(tr, te, thresholds=(0, 2, 4, 6, 10)):
    u = lambda s: s.to_numpy(dtype=np.uint64)
    d_te, j_te = min_hamming_idx(u(te.dh), u(te.dhm), u(tr.dh))
    d_tr, j_tr = min_hamming_idx(u(tr.dh), u(tr.dhm), u(tr.dh), exclude_self=True)
    y = tr.target.to_numpy()
    same = y[j_tr] == y

    print("\n[A] KEMBARAN vs AMBANG (jarak ke tetangga TERDEKAT, tanpa rantai union-find)")
    rows = []
    for t in thresholds:
        rows.append({"ambang": t,
                     "test punya kembaran di train": f"{(d_te <= t).mean() * 100:.1f}%",
                     "train punya kembaran di train lain": f"{(d_tr <= t).mean() * 100:.1f}%",
                     **{f"{c} punya kembaran": f"{((d_tr <= t) & (y == i)).sum() / (y == i).sum() * 100:.1f}%"
                        for i, c in enumerate(CLASSES)}})
    print(pd.DataFrame(rows).to_string(index=False))

    print("\n[B] KEANDALAN MENYALIN LABEL KEMBARAN (train terhadap tetangga train terdekatnya)")
    rows = []
    for t in thresholds:
        m = d_tr <= t
        r = {"ambang": t, "n": int(m.sum()),
             "akurasi salin": f"{same[m].mean() * 100:.1f}%" if m.sum() else "-"}
        for i, c in enumerate(CLASSES):
            mm = m & (y == i)
            r[f"recall {c}"] = f"{same[mm].mean() * 100:.1f}% (n={int(mm.sum())})" if mm.sum() else "-"
        rows.append(r)
    print(pd.DataFrame(rows).to_string(index=False))
    print("    Dibaca: jika akurasi salin ~100% di ambang kecil tetapi turun di ambang besar, "
          "kembaran 'sejati' dan 'mirip saja' perlu dibedakan.")
    return dict(d_te=d_te, j_te=j_te, d_tr=d_tr, j_tr=j_tr)


# --------------------------------------------------------------------------- [C]
def joint_groups(tr, te):
    u = lambda s: s.to_numpy(dtype=np.uint64)
    dh = np.r_[u(tr.dh), u(te.dh)]
    dhm = np.r_[u(tr.dhm), u(te.dhm)]
    pairs = hamming_pairs(dh, dhm, np.ones(len(dh), dtype=bool), HAMMING_THR)
    return union_groups(len(dh), pairs)


def adversarial_grouped(tr, te, groups_all):
    y = np.r_[np.zeros(len(tr)), np.ones(len(te))]
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
    compression = {"nbytes", "bpp", "is_jpeg", "is_png"}
    sets = {
        "semua fitur": META_COLS + CONTENT_COLS,
        "tanpa fitur kompresi (byte/bpp/format)": [c for c in META_COLS + CONTENT_COLS if c not in compression],
        "hanya ukuran (w,h,ar,mp)": ["w", "h", "ar", "mp"],
        "hanya statistik isi": CONTENT_COLS,
    }
    print("\n[C] ADVERSARIAL VALIDATION BERKELOMPOK (kembaran tidak boleh terbelah antar lipatan)")
    for name, cols in sets.items():
        both = pd.concat([tr[cols], te[cols]], ignore_index=True)
        both = both.fillna(both.median())
        p = cross_val_predict(HistGradientBoostingClassifier(max_iter=120, random_state=SEED),
                              both, y, cv=cv, groups=groups_all, method="predict_proba")[:, 1]
        print(f"    AUC {roc_auc_score(y, p):.4f}  <- {name}")
    cols = ["w", "h", "nbytes", "bpp", "noise", "bright", "contrast", "sat", "blur"]
    med = pd.DataFrame({"train": tr[cols].median(), "test": te[cols].median()})
    print("    Median train vs test:")
    print(med.to_string(float_format=lambda v: f"{v:.5f}").replace("\n", "\n      "))
    fmt = pd.DataFrame({"train": tr.fmt.value_counts(normalize=True),
                        "test": te.fmt.value_counts(normalize=True)}).fillna(0)
    print("    Porsi format berkas:")
    print((fmt * 100).round(1).to_string().replace("\n", "\n      "))


# --------------------------------------------------------------------------- [D][E]
def size_tables(tr, te, top=8):
    key = lambda d: d.w.astype(int).astype(str) + "x" + d.h.astype(int).astype(str)
    tab = pd.crosstab(key(tr).rename("ukuran"), tr.label)
    tab["test"] = key(te).value_counts()
    tab = tab.fillna(0).astype(int)
    tab["total"] = tab.sum(axis=1)
    print("\n[D] UKURAN GAMBAR PALING UMUM (jumlah per kelas dan di test)")
    print(tab.sort_values("total", ascending=False).head(top).drop(columns="total").to_string())
    small = lambda d: (np.minimum(d.w, d.h) <= 256)
    print("    Porsi gambar dengan sisi pendek <= 256 px:")
    for c in CLASSES:
        print(f"      train/{c:<8}: {small(tr[tr.label == c]).mean() * 100:5.1f}%")
    print(f"      test           : {small(te).mean() * 100:5.1f}%   "
          f"(porsi Lainnya di train = {(tr.label == 'Lainnya').mean() * 100:.1f}%)")


def metadata_per_class(tr):
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=SEED)
    X = tr[META_COLS].fillna(tr[META_COLS].median())
    pred = cross_val_predict(HistGradientBoostingClassifier(max_iter=120, random_state=SEED),
                             X, tr.target, cv=cv, groups=tr.group)
    print("\n[E] METADATA SAJA - precision/recall per kelas (CV berkelompok)")
    print(classification_report(tr.target, pred, target_names=CLASSES, digits=3))


# --------------------------------------------------------------------------- [F]
def pair_sheet(tr, te, d_te, j_te, lo, hi, path, n=8, cell=170):
    idx = np.where((d_te >= lo) & (d_te <= hi))[0]
    if len(idx) == 0:
        return
    rng = np.random.default_rng(SEED)
    idx = rng.permutation(idx)[:n]
    cols = 4
    rows = int(np.ceil(len(idx) / 2))
    sheet = Image.new("RGB", (cols * cell, rows * (cell + 16)), (255, 255, 255))
    dr = ImageDraw.Draw(sheet)
    for k, i in enumerate(idx):
        r, c0 = k // 2, (k % 2) * 2
        j = int(j_te[i])
        for off, p, cap in ((0, te.path.iloc[i], "TEST"),
                            (1, tr.path.iloc[j], f"train: {tr.label.iloc[j]} (d={int(d_te[i])})")):
            try:
                with Image.open(p) as im:
                    im = ImageOps.fit(ImageOps.exif_transpose(im).convert("RGB"), (cell, cell))
                sheet.paste(im, ((c0 + off) * cell, r * (cell + 16) + 16))
            except Exception:
                pass
            dr.text(((c0 + off) * cell + 4, r * (cell + 16) + 2), cap, fill=(200, 0, 0))
    sheet.save(path)


# --------------------------------------------------------------------------- main
def run_followup(res=None, out_dir=None, n_workers=4):
    res = res or globals().get("RESULT")
    assert res is not None, "Berikan hasil run_diagnosis(): run_followup(RESULT)"
    tr, te = ensure_hashes(res["train"], n_workers), ensure_hashes(res["test"], n_workers)
    out = Path(out_dir or (CFG.OUT_ROOT / "diagnosis"))
    out.mkdir(parents=True, exist_ok=True)

    tw = twin_sensitivity(tr, te)
    groups_all = joint_groups(tr, te)
    adversarial_grouped(tr, te, groups_all)
    size_tables(tr, te)
    metadata_per_class(tr)
    for lo, hi, name in ((0, 0, "d0_identik"), (1, 3, "d1-3"), (4, 6, "d4-6")):
        pair_sheet(tr, te, tw["d_te"], tw["j_te"], lo, hi, out / f"sheet_test_twins_{name}.png")
    print(f"\n[F] Contact sheet pasangan test-train disimpan di {out} (sheet_test_twins_*.png).")
    print("    BACA: apakah pasangan d=4-6 benar-benar foto yang sama, atau hanya mirip? "
          "Itu menentukan seberapa besar 42% itu nyata.")
    return tw


if __name__ == "__main__":
    FOLLOWUP = run_followup()


[A] KEMBARAN vs AMBANG (jarak ke tetangga TERDEKAT, tanpa rantai union-find)
 ambang test punya kembaran di train train punya kembaran di train lain Aman punya kembaran Rusak punya kembaran Lainnya punya kembaran
      0                        15.1%                              19.3%               14.4%                36.2%                   1.9%
      2                        25.2%                              28.0%               23.7%                43.6%                   6.4%
      4                        32.4%                              33.0%               29.2%                47.3%                   9.2%
      6                        39.6%                              37.3%               34.3%                50.1%                  10.6%
     10                        60.2%                              53.9%               52.6%                62.2%                  21.0%

[B] KEANDALAN MENYALIN LABEL KEMBARAN (train terhadap tetangga train terdekatnya)
 ambang    n akurasi sa

**5.3** apakah test berisi versi terdegradasi (diperkecil / dikompres ulang) dari gambar train?

In [7]:
# =============================================================================
# DATASCAPE 2026 - DIAGNOSIS LANJUTAN 2
# Jalankan SETELAH datascape_diagnosis_2.py (butuh RESULT dan FOLLOWUP).
#
# Pertanyaan: ukuran gambar test sangat berbeda dari train (224x224 = 17% test
# vs 2% train; 512x512 = 1.5% test vs 20% train). Apakah test berisi VERSI
# TERDEGRADASI (diperkecil / dikompres ulang) dari gambar yang ada di train,
# atau sumber yang benar-benar baru?
#   [G] Pasangan test-train kembar: ukuran, byte, format berubah ke arah mana?
#   [H] Per ukuran test: berapa persen punya kembaran, dan label kembarannya apa?
#   [I] Komposisi label di antara test yang punya kembaran vs prior train
# =============================================================================
import numpy as np
import pandas as pd

try:
    run_followup
except NameError:
    from datascape_diagnosis_2 import *  # noqa: F401,F403
    from datascape_diagnosis_2 import run_followup  # noqa: F401


def _size_key(df):
    return df.w.astype(int).astype(str) + "x" + df.h.astype(int).astype(str)


def run_followup2(res=None, tw=None, thr=4, top_sizes=6):
    res = res or globals().get("RESULT")
    tw = tw or globals().get("FOLLOWUP")
    assert res is not None and tw is not None, "Jalankan skrip 1 dan 2 dulu (butuh RESULT dan FOLLOWUP)."
    tr, te = res["train"], res["test"]
    d, j = tw["d_te"], tw["j_te"]
    m = d <= thr
    classes = ["Aman", "Rusak", "Lainnya"]

    tj = tr.iloc[j].reset_index(drop=True)          # kembaran terdekat di train untuk tiap citra test
    te = te.reset_index(drop=True)
    area_ratio = (te.w * te.h) / (tj.w * tj.h).clip(lower=1)
    byte_ratio = te.nbytes / tj.nbytes.clip(lower=1)
    same_size = (_size_key(te) == _size_key(tj)).to_numpy()

    print(f"\n[G] PASANGAN TEST-TRAIN KEMBAR (jarak dHash <= {thr}): {int(m.sum()):,} dari {len(te):,} citra test")
    print(f"    ukuran persis sama                : {same_size[m].mean() * 100:5.1f}%")
    print(f"    test LEBIH KECIL dari kembarannya : {(area_ratio[m] < 0.99).mean() * 100:5.1f}%")
    print(f"    test LEBIH BESAR                  : {(area_ratio[m] > 1.01).mean() * 100:5.1f}%")
    print(f"    median rasio luas test/train      : {area_ratio[m].median():.3f}")
    print(f"    median rasio byte test/train      : {byte_ratio[m].median():.3f}")
    print("    Perubahan format (baris = format train, kolom = format test):")
    print(pd.crosstab(tj.fmt[m].rename("train"), te.fmt[m].rename("test")).to_string().replace("\n", "\n      "))

    sizes = _size_key(te)
    top = sizes.value_counts().head(top_sizes).index.tolist()
    bucket = sizes.where(sizes.isin(top), "lain-lain")
    prior = tr.label.value_counts(normalize=True)
    rows = []
    for name in top + ["lain-lain"]:
        sel = (bucket == name).to_numpy()
        tw_sel = sel & m
        lab = tj.label[tw_sel].value_counts(normalize=True) if tw_sel.any() else pd.Series(dtype=float)
        rows.append({"ukuran test": name, "n test": int(sel.sum()),
                     "% punya kembaran": f"{tw_sel.sum() / max(sel.sum(), 1) * 100:.0f}%",
                     **{f"kembaran={c}": f"{lab.get(c, 0) * 100:.0f}%" for c in classes}})
    print("\n[H] PER UKURAN TEST: berapa yang punya kembaran, dan label kembarannya")
    print(pd.DataFrame(rows).to_string(index=False))
    print("    (hanya dihitung pada citra yang punya kembaran; selebihnya tidak diketahui)")

    lab_all = tj.label[m].value_counts(normalize=True)
    print("\n[I] KOMPOSISI LABEL KEMBARAN TEST vs PRIOR TRAIN")
    print(pd.DataFrame({"prior train": prior, "kembaran test": lab_all}).reindex(classes)
          .fillna(0).mul(100).round(1).to_string().replace("\n", "\n    "))
    print("    Catatan: komposisi ini bias ke kelas yang sering digandakan (lihat [A]: Rusak 50% punya kembaran, "
          "Lainnya 11%); bukan estimasi prior test yang sebenarnya.")
    return dict(pair_mask=m, area_ratio=area_ratio, byte_ratio=byte_ratio)


if __name__ == "__main__":
    FOLLOWUP2 = run_followup2()


[G] PASANGAN TEST-TRAIN KEMBAR (jarak dHash <= 4): 990 dari 3,057 citra test
    ukuran persis sama                :  52.0%
    test LEBIH KECIL dari kembarannya :  20.9%
    test LEBIH BESAR                  :  25.4%
    median rasio luas test/train      : 1.000
    median rasio byte test/train      : 0.997
    Perubahan format (baris = format train, kolom = format test):
test   JPEG  PNG  WEBP
      train                 
      JPEG    704   27    73
      PNG       1   11     0
      WEBP     82    2    90

[H] PER UKURAN TEST: berapa yang punya kembaran, dan label kembarannya
ukuran test  n test % punya kembaran kembaran=Aman kembaran=Rusak kembaran=Lainnya
    224x224     520              22%          100%             0%               0%
    640x640     347              69%           27%            72%               1%
    675x900     280              23%          100%             0%               0%
    416x416     193              20%           49%            51%               

## 6. Preprocessing berbasis diagnosis
Mengubah hasil diagnosis menjadi kolom/aturan yang dipakai training:

| Kolom | Arti | Dipakai untuk |
|---|---|---|
| `fold` | fold **berkelompok** (StratifiedGroupKFold) | validasi tanpa kebocoran kembaran |
| `noisy` | punya kembaran (dHash ≤ `NOISE_HAM`) berlabel beda | dikeluarkan dari training |
| `gw` | `ukuran_kelompok^-GROUP_W_POWER` | bobot sampler |
| `shift_w` | density-ratio train→test dari adversarial validation | Macro F1 berbobot-shift (OOF, bias) |
| `twin_label`, `twin_d` | label & jarak kembaran terdekat di train (≤ `TWIN_HAM`) | twin blend (OOF dan test) |


In [8]:
from collections import Counter
# ======================= 6. Diagnosis -> preprocessing =======================
tr_m, te_m = RESULT["train"].copy().reset_index(drop=True), RESULT["test"].copy().reset_index(drop=True)
d_te, j_te, d_tr, j_tr = (np.asarray(FOLLOWUP[k]) for k in ("d_te", "j_te", "d_tr", "j_tr"))
u64 = lambda s: s.to_numpy(dtype=np.uint64)

# (a) lampirkan path cache + rel dari section 3-4 (urutan baris mengikuti manifest diagnosis, indeks j_* tetap valid)
tr_m["fpath"] = tr_m.path.map(dict(zip(train_df.path, train_df.fpath)))
tr_m["rel"]   = tr_m.path.map(dict(zip(train_df.path, train_df.rel)))
te_m["fpath"] = te_m.path.map(dict(zip(test_df.path, test_df.fpath)))
assert tr_m.rel.notna().all(), "path train di diagnosis tidak cocok dgn section 3 (cek IMG_EXTS)"
assert len(tr_m) == len(train_df) and len(te_m) == len(test_df)
assert te_m.filename.is_unique

# (a2) Kelompok kembaran SIMETRIS + fold diulang.
# Jarak dHash+cermin di diagnosis tidak simetris (versi cermin hanya dipakai di sisi i), jadi sepasang gambar bisa dianggap
# kembar dari i tetapi tidak dari j -> tidak digabung -> terbelah antar fold. Di sini jarak = min(asli-asli, cermin_i-asli_j, asli_i-cermin_j).
def sym_pairs(dh, dhm, thr, chunk=512):
    pairs, n = [], len(dh)
    for i0 in range(0, n, chunk):
        a, am = dh[i0:i0 + chunk, None], dhm[i0:i0 + chunk, None]
        d = np.minimum(np.minimum(popcount64(a ^ dh[None, :]), popcount64(am ^ dh[None, :])),
                       popcount64(a ^ dhm[None, :]))
        ii, jj = np.where(d <= thr)
        for p, q in zip(ii, jj):
            i, j = i0 + int(p), int(q)
            if i < j: pairs.append((i, j, int(d[p, q])))
    return pairs

pairs_g = sym_pairs(u64(tr_m.dh), u64(tr_m.dhm), CFG.GROUP_HAM)
tr_m["group"] = union_groups(len(tr_m), [(i, j) for i, j, _ in pairs_g])
tr_m["fold"] = -1
for _f, (_, _v) in enumerate(StratifiedGroupKFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
                             .split(tr_m, tr_m.target, tr_m.group)):
    tr_m.loc[tr_m.index[_v], "fold"] = _f
for _f in range(CFG.N_FOLDS):
    assert not (set(tr_m.loc[tr_m.fold != _f, "group"]) & set(tr_m.loc[tr_m.fold == _f, "group"])), f"kelompok terbelah di fold {_f}"
log(f"Kelompok kembaran simetris (dHash<={CFG.GROUP_HAM}): {tr_m.group.nunique()} kelompok untuk {len(tr_m)} gambar | "
    f"kelompok terbesar {int(tr_m.group.value_counts().max())} | fold berkelompok dibuat ulang")

# (b) label noise: pasangan kembaran hampir identik (dHash<=NOISE_HAM, termasuk cermin) dgn label berbeda
y_tr = tr_m.target.to_numpy()
noisy = np.zeros(len(tr_m), dtype=bool)
conf = Counter()
for i, j, d in pairs_g:
    if d <= CFG.NOISE_HAM and y_tr[i] != y_tr[j]:
        noisy[i] = noisy[j] = True
        conf[" vs ".join(sorted((CLASSES[y_tr[i]], CLASSES[y_tr[j]])))] += 1
tr_m["noisy"] = noisy
log(f"Label noise (kembaran dHash<={CFG.NOISE_HAM} beda label): {int(noisy.sum())} gambar dikeluarkan dari training "
    f"| pasangan: {dict(conf)} | per kelas: {tr_m[tr_m.noisy].label.value_counts().to_dict()}")

# (c) bobot kelompok (kelompok raksasa = duplikat / gambar polos tidak boleh mendominasi)
tr_m["gsize"] = tr_m.groupby("group")["group"].transform("size")
tr_m["gw"] = tr_m.gsize.astype(float) ** (-CFG.GROUP_W_POWER)

# (d) bobot pergeseran train->test (density ratio) dari adversarial validation BERKELOMPOK
groups_all = joint_groups(tr_m, te_m)
_cols = META_COLS + CONTENT_COLS
_both = pd.concat([tr_m[_cols], te_m[_cols]], ignore_index=True); _both = _both.fillna(_both.median())
_yb = np.r_[np.zeros(len(tr_m)), np.ones(len(te_m))]
_p = cross_val_predict(HistGradientBoostingClassifier(max_iter=120, random_state=CFG.SEED), _both, _yb,
                       cv=StratifiedGroupKFold(5, shuffle=True, random_state=CFG.SEED), groups=groups_all,
                       method="predict_proba")[:, 1]
log(f"Adversarial AUC (berkelompok, semua fitur) = {roc_auc_score(_yb, _p):.4f}")
_p = np.clip(_p[:len(tr_m)], 0.02, 0.98)
_ratio = _p / (1 - _p) * (len(tr_m) / len(te_m))                  # p_test(x) / p_train(x)
_w = np.clip(_ratio ** CFG.SHIFT_W_POWER, *CFG.SHIFT_W_CLIP); _w = _w / _w.mean()
tr_m["shift_w"] = _w
ess = _w.sum() ** 2 / (_w ** 2).sum()
est_prior = (pd.Series(_w).groupby(tr_m.label.values).sum() / _w.sum()).reindex(CLASSES)
log(f"Bobot shift: min {_w.min():.2f} | median {np.median(_w):.2f} | max {_w.max():.2f} | ESS {ess:.0f}/{len(_w)}")
log(f"Prior kelas train {tr_m.label.value_counts(normalize=True).reindex(CLASSES).round(3).to_dict()} "
    f"-> setelah dibobot shift (perkiraan kasar prior test) {est_prior.round(3).to_dict()}")

# (e) kembaran berlabel: utk test (dari train) dan utk train (simulasi OOF jujur: kembaran satu kelompok = satu fold)
assert (tr_m.fold.to_numpy()[j_tr] == tr_m.fold.to_numpy())[d_tr <= CFG.GROUP_HAM].all(), "kembaran (twin) terbelah antar fold"
ok_te = (d_te <= CFG.TWIN_HAM) & (~noisy[j_te])
ok_tr = (d_tr <= CFG.TWIN_HAM) & (~noisy[j_tr])
te_m["twin_d"], te_m["twin_label"] = d_te, np.where(ok_te, y_tr[j_te], -1)
tr_m["twin_d"], tr_m["twin_label"] = d_tr, np.where(ok_tr, y_tr[j_tr], -1)
log(f"Kembaran berlabel (dHash<={CFG.TWIN_HAM}): test {int(ok_te.sum())}/{len(te_m)} ({ok_te.mean()*100:.1f}%) | "
    f"train {int(ok_tr.sum())}/{len(tr_m)} | label kembaran test: {pd.Series(te_m.twin_label[te_m.twin_label>=0]).map(dict(enumerate(CLASSES))).value_counts().to_dict()}")

train_df, test_df = tr_m, te_m
train_df.to_csv(CFG.OUTPUTS_DIR / "fold_assignments_grouped.csv", index=False,
                columns=["rel", "path", "label", "target", "group", "fold", "noisy", "gsize", "shift_w", "twin_d", "twin_label"])
print(train_df.groupby(["fold", "label"]).size().unstack())
print("Training per fold (tanpa noisy):", (train_df[~train_df.noisy].groupby("fold").size()).to_dict())
log(f"Persiapan data selesai: {elapsed_h()*60:.1f} menit")


2026-10-02 09:57:36,496 | INFO | Kelompok kembaran simetris (dHash<=6): 9331 kelompok untuk 12224 gambar | kelompok terbesar 18 | fold berkelompok dibuat ulang
2026-10-02 09:57:36,502 | INFO | Label noise (kembaran dHash<=2 beda label): 176 gambar dikeluarkan dari training | pasangan: {'Aman vs Lainnya': 9, 'Aman vs Rusak': 127} | per kelas: {'Aman': 86, 'Rusak': 85, 'Lainnya': 5}
2026-10-02 09:58:09,019 | INFO | Adversarial AUC (berkelompok, semua fitur) = 0.8671
2026-10-02 09:58:09,027 | INFO | Bobot shift: min 0.40 | median 0.65 | max 7.05 | ESS 7246/12224
2026-10-02 09:58:09,030 | INFO | Prior kelas train {'Aman': 0.72, 'Rusak': 0.246, 'Lainnya': 0.035} -> setelah dibobot shift (perkiraan kasar prior test) {'Aman': 0.634, 'Rusak': 0.331, 'Lainnya': 0.035}
2026-10-02 09:58:09,036 | INFO | Kembaran berlabel (dHash<=4): test 968/3057 (31.7%) | train 3844/12224 | label kembaran test: {'Aman': 673, 'Rusak': 294, 'Lainnya': 1}
label  Aman  Lainnya  Rusak
fold                       
0    

## 7. Dataset, Augmentasi, Sampler, Loss
Dibanding eva001: (1) **degradasi test-like** — `Downscale` (perkecil lalu perbesar) + kompresi JPEG **atau WebP** berkualitas rendah, diterapkan ke semua kelas; (2) sampler = bobot kelas parsial × **bobot kelompok**, dengan jumlah sampel/epoch tetap (`EPOCH_SAMPLES`); (3) gambar `noisy` tidak ikut training.

In [9]:
def make_transforms(S, mean, std):
    train_tf = A.Compose([
        A.RandomResizedCrop(size=(S, S), scale=CFG.RRC_SCALE, ratio=(0.8, 1.25)),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.06, 0.06), rotate=(-12, 12), p=0.5),
        A.Downscale(scale_range=CFG.DOWNSCALE_RANGE,
                    interpolation_pair=dict(downscale=cv2.INTER_AREA, upscale=cv2.INTER_LINEAR), p=CFG.DOWNSCALE_P),
        A.OneOf([
            A.ColorJitter(0.35, 0.35, 0.35, 0.05),
            A.RandomBrightnessContrast(0.35, 0.35),
            A.RandomGamma(gamma_limit=(60, 160)),
            A.CLAHE(clip_limit=(1, 4)),
        ], p=0.8),
        A.OneOf([A.GaussianBlur(blur_limit=(3, 7)), A.MotionBlur(blur_limit=(3, 7))], p=0.25),
        A.OneOf([A.GaussNoise(std_range=(0.02, 0.10)), A.ISONoise()], p=0.30),
        A.OneOf([A.ImageCompression(compression_type="jpeg", quality_range=CFG.COMPRESS_Q),
                 A.ImageCompression(compression_type="webp", quality_range=CFG.COMPRESS_Q)], p=CFG.COMPRESS_P),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])
    eval_tf = A.Compose([A.Resize(S, S), A.Normalize(mean=mean, std=std), ToTensorV2()])   # seluruh gambar, tanpa crop
    return train_tf, eval_tf

class PkgDataset(Dataset):
    def __init__(self, df, transform, with_target=True):
        self.paths = df.fpath.tolist(); self.tf = transform
        self.targets = df.target.astype(int).tolist() if with_target else [0] * len(df)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        img = cv2.imread(p) if isinstance(p, str) else None
        img = np.zeros((256, 256, 3), np.uint8) if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.tf(image=img)["image"], self.targets[i]

def worker_init(worker_id):
    info = torch.utils.data.get_worker_info()
    s = (torch.initial_seed() + worker_id) % (2 ** 32)
    random.seed(s); np.random.seed(s)
    ds = getattr(info.dataset, "dataset", info.dataset)   # Subset -> dataset asli
    try: ds.tf.set_random_seed(int(s))
    except Exception: pass

class SoftTargetCE(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.register_buffer("cw", class_weights if class_weights is not None else torch.ones(CFG.NUM_CLASSES))
    def forward(self, logits, soft_targets):
        per = -(soft_targets * F.log_softmax(logits, dim=-1)).sum(-1)
        return (per * (soft_targets * self.cw).sum(-1)).mean()

def loss_class_weights(df):
    cnt = df.target.value_counts().sort_index().values.astype(float)
    w = (len(df) / (CFG.NUM_CLASSES * cnt)) ** CFG.LOSS_CW_POWER
    return torch.tensor(w / w.mean(), dtype=torch.float32)

def epoch_order(tr, seed, global_bs):
    """Urutan sampel per-epoch yang DETERMINISTIK (agar resume tengah-epoch presisi).
    Bobot = n_kelas^-SAMPLER_POWER x ukuran_kelompok^-GROUP_W_POWER; jumlah sampel tetap = min(len(tr), EPOCH_SAMPLES)."""
    freq = tr.target.value_counts().to_dict()
    w = tr.target.map(lambda t: freq[t] ** (-CFG.SAMPLER_POWER)).values * tr.gw.values
    gen = torch.Generator().manual_seed(seed)
    order = list(WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double),
                                       num_samples=min(len(tr), CFG.EPOCH_SAMPLES), replacement=True, generator=gen))
    return order[:(len(order) // global_bs) * global_bs]

_t = train_df[~train_df.noisy]
_cnt = _t.target.value_counts()
_mass = pd.Series(_t.target.map(lambda k: _cnt[k] ** (-CFG.SAMPLER_POWER)).values * _t.gw.values).groupby(_t.target.values).sum()
_mass = (_mass / _mass.sum()).sort_index()
log(f"Proporsi kelas asli {dict(zip(CFG.CLASSES, (_cnt.sort_index()/_cnt.sum()).round(3).values))} -> setelah sampler (bobot kelas x kelompok) "
    f"{dict(zip(CFG.CLASSES, _mass.round(3).values))} | training {len(_t)} gambar, {CFG.EPOCH_SAMPLES} sampel/epoch")


2026-10-02 09:58:09,269 | INFO | Proporsi kelas asli {'Aman': np.float64(0.723), 'Rusak': np.float64(0.242), 'Lainnya': np.float64(0.035)} -> setelah sampler (bobot kelas x kelompok) {'Aman': np.float64(0.562), 'Rusak': np.float64(0.301), 'Lainnya': np.float64(0.137)} | training 12048 gambar, 8700 sampel/epoch


## 8. Training Loop EVA02 — fp16, 2 GPU, resume presisi, pengaman waktu
- `fold{f}_best.pt` : bobot **EMA** terbaik (val Macro F1) → dipakai untuk OOF & test.
- `fold{f}_last.pt` / `fold{f}_mid.pt` : state lengkap (model+EMA+optimizer+scaler) akhir-epoch / tiap `SAVE_EVERY_MIN` menit.
- `fold{f}_done.json` : penanda fold selesai (dilewati di sesi berikutnya).
- Resume memilih progres **terjauh** dari output sesi ini + `PREV_OUTPUT_DIRS`; urutan sampel deterministik sehingga lanjut dari batch yang tepat.

In [10]:
def layerwise_lr_param_groups(model, base_lr, decay, weight_decay):
    def depth_of(name):
        for key in ("blocks.", "stages.", "layers."):
            if key in name:
                try: return int(name.split(key)[1].split(".")[0]) + 1
                except Exception: pass
        if any(k in name for k in ("patch_embed", "stem", "cls_token", "pos_embed")):
            return 0
        return None
    depths = [d for d in (depth_of(n) for n, _ in model.named_parameters()) if d is not None]
    max_d = (max(depths) if depths else 0) + 1
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        d = depth_of(n); d = max_d if d is None else d
        wd = 0.0 if p.ndim <= 1 or "bias" in n else weight_decay
        groups.setdefault((d, wd), dict(params=[], lr=base_lr * (decay ** (max_d - d)), weight_decay=wd))["params"].append(p)
    return list(groups.values())

@torch.no_grad()
def predict_probs(model, loader, tta=False):
    model.eval(); probs = []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            p = F.softmax(model(x).float(), -1)
            if tta:
                p = (p + F.softmax(model(torch.flip(x, dims=[3])).float(), -1)) / 2
        probs.append(p.float().cpu().numpy())
    return np.concatenate(probs) if probs else np.zeros((0, CFG.NUM_CLASSES))

EPOCH_TIME_H = None
def time_left_ok(margin=1.15):
    est = EPOCH_TIME_H if EPOCH_TIME_H is not None else CFG.FIRST_EPOCH_EST_H
    return elapsed_h() + est * margin < CFG.TIME_BUDGET_H
def time_for_one_more_interval_ok(margin=1.3):
    return elapsed_h() + (CFG.SAVE_EVERY_MIN / 60.0) * margin < CFG.TIME_BUDGET_H

def build_model():
    base = timm.create_model(CFG.TIMM_ID, pretrained=True, num_classes=CFG.NUM_CLASSES,
                             img_size=CFG.IMG_SIZE, drop_path_rate=CFG.DROP_PATH).to(DEVICE)
    if CFG.GRAD_CKPT:
        try: base.set_grad_checkpointing()
        except Exception as e: log(f"grad checkpointing tidak tersedia: {e}")
    return base

def _save_ckpt(path, base, ema, opt, scaler, epoch, best_f1, batches_done=None, light=False):
    """light=True -> hanya BOBOT EMA + metadata (utk best; inferensi cuma butuh ini)."""
    if light:
        state = dict(model=ema.module.state_dict(), is_ema=True)
    else:
        state = dict(model=base.state_dict(), ema=ema.module.state_dict(),
                     opt=opt.state_dict(), scaler=scaler.state_dict())
    state.update(epoch=int(epoch), best_f1=float(best_f1), timm_id=CFG.TIMM_ID, img_size=CFG.IMG_SIZE)
    if batches_done is not None: state["batches_done"] = int(batches_done)
    tmp = Path(str(path) + ".tmp")
    if tmp.exists():
        try: tmp.unlink()
        except Exception: pass
    try:
        free_gb = shutil.disk_usage(tmp.parent).free / 1e9
        if free_gb < 3.0: log(f"[save_ckpt] PERINGATAN: disk tersisa {free_gb:.2f}GB")
        torch.save(state, tmp); tmp.replace(path)
    except (RuntimeError, OSError) as e:
        log(f"[save_ckpt] GAGAL simpan {path.name}: {e} -> dilewati, training lanjut")
        if tmp.exists():
            try: tmp.unlink()
            except Exception: pass
        return False
    json.dump(dict(epoch=int(epoch), batches_done=int(batches_done or 0), best_f1=float(best_f1),
                   is_mid=batches_done is not None), open(str(path) + ".meta.json", "w"))
    return True

def _read_meta(p):
    mp = Path(str(p) + ".meta.json")
    if mp.exists():
        try: return json.load(open(mp))
        except Exception: pass
    return None

def _cleanup(path):
    for p in (Path(path), Path(str(path) + ".meta.json")):
        if p.exists(): p.unlink()

def fold_done_info(fold):
    p = find_in_roots(f"models/{CFG.MODEL_NAME}/fold{fold}_done.json")
    return json.load(open(p)) if p is not None else None

def train_one_fold(fold):
    global EPOCH_TIME_H
    info = fold_done_info(fold)
    if info is not None:
        log(f"[fold{fold}] sudah selesai (best_f1={info['best_f1']:.5f}) -> dilewati"); return info["best_f1"], True
    seed_everything(CFG.SEED + fold)
    base = build_model()
    net = nn.DataParallel(base) if N_GPUS > 1 else base
    dcfg = timm.data.resolve_model_data_config(base)
    train_tf, eval_tf = make_transforms(CFG.IMG_SIZE, dcfg["mean"], dcfg["std"])

    tr = train_df[(train_df.fold != fold) & (~train_df.noisy)].reset_index(drop=True)   # noisy: tidak ikut training
    va = train_df[train_df.fold == fold].reset_index(drop=True)
    tr_ds = PkgDataset(tr, train_tf)
    vl = DataLoader(PkgDataset(va, eval_tf), batch_size=GLOBAL_BS * 2, shuffle=False,
                    num_workers=CFG.NUM_WORKERS, pin_memory=True)

    mixup_on  = Mixup(mixup_alpha=CFG.MIXUP_ALPHA, cutmix_alpha=CFG.CUTMIX_ALPHA, prob=CFG.MIXCUT_P,
                      label_smoothing=CFG.LABEL_SMOOTHING, num_classes=CFG.NUM_CLASSES)
    mixup_off = Mixup(mixup_alpha=0.0, cutmix_alpha=0.0, prob=0.0,
                      label_smoothing=CFG.LABEL_SMOOTHING, num_classes=CFG.NUM_CLASSES)
    criterion = SoftTargetCE(loss_class_weights(tr).to(DEVICE))
    opt = torch.optim.AdamW(layerwise_lr_param_groups(base, CFG.LR, CFG.LAYER_DECAY, CFG.WEIGHT_DECAY))

    raw_batches = min(len(tr), CFG.EPOCH_SAMPLES) // GLOBAL_BS
    steps_per_epoch = max(1, raw_batches // ACCUM)
    total, warm = CFG.EPOCHS * steps_per_epoch, CFG.WARMUP_EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / max(1, warm) if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16_SCALER)
    ema_decay = float(min(0.9995, max(0.99, 1 - 2.0 / steps_per_epoch)))   # horizon ≈ setengah epoch
    ema = ModelEmaV2(base, decay=ema_decay)
    log(f"[fold{fold}] train={len(tr)} val={len(va)} | steps/epoch={steps_per_epoch} | EMA decay={ema_decay:.4f}")

    mdir = CFG.MODELS_DIR / CFG.MODEL_NAME; mdir.mkdir(parents=True, exist_ok=True)
    best_ckpt, last_ckpt, mid_ckpt = (mdir / f"fold{fold}_{k}.pt" for k in ("best", "last", "mid"))

    # ---------- Resume: progres TERJAUH dari semua root ----------
    best_choice = None
    if CFG.RESUME:
        for r in all_roots():
            for kind in ("mid", "last", "best"):
                p = Path(r) / f"models/{CFG.MODEL_NAME}/fold{fold}_{kind}.pt"
                meta = _read_meta(p) if p.exists() else None
                if meta is None: continue
                prog = (meta["epoch"] + (0 if meta["is_mid"] else 1), meta["batches_done"] if meta["is_mid"] else 0)
                if best_choice is None or prog > best_choice[1]:
                    best_choice = (p, prog, meta)

    start_epoch, resume_off, best_f1, patience = 0, 0, -1.0, 0
    if best_choice is not None:
        p, prog, meta = best_choice
        st = torch.load(p, map_location="cpu", weights_only=False)
        base.load_state_dict(st["model"])
        ema.module.load_state_dict(st["ema"] if "ema" in st else st["model"])
        if "opt" in st:
            try: opt.load_state_dict(st["opt"])
            except Exception as e: log(f"[fold{fold}] optimizer state gagal dimuat ({e}); mulai baru")
        if "scaler" in st:
            try: scaler.load_state_dict(st["scaler"])
            except Exception: pass
        best_f1 = float(meta.get("best_f1", -1.0))
        start_epoch, resume_off = prog
        for _ in range(start_epoch * steps_per_epoch + resume_off // ACCUM): sched.step()
        log(f"[fold{fold}] RESUME dari {p} -> epoch={start_epoch}, batch={resume_off}/{raw_batches}, best_f1={best_f1:.5f}")
        del st; gc.collect()

    for epoch in range(start_epoch, CFG.EPOCHS):
        if not time_left_ok():
            log(f"[fold{fold}] STOP sebelum ep{epoch}: sisa waktu kurang ({elapsed_h():.2f}h/{CFG.TIME_BUDGET_H}h). Resume di sesi berikutnya.")
            return best_f1, False

        batch_off = resume_off if epoch == start_epoch else 0
        order = epoch_order(tr, CFG.SEED * 1009 + fold * 97 + epoch, GLOBAL_BS)
        remaining = order[batch_off * GLOBAL_BS:]
        tl = (DataLoader(Subset(tr_ds, remaining), batch_size=GLOBAL_BS, shuffle=False, num_workers=CFG.NUM_WORKERS,
                         pin_memory=True, drop_last=True, worker_init_fn=worker_init)
              if len(remaining) >= GLOBAL_BS else [])
        mixup_fn = mixup_off if (CFG.MIXUP_OFF_LAST_EPOCH and epoch == CFG.EPOCHS - 1) else mixup_on

        net.train(); t0 = time.time(); run_loss, n_loss = 0.0, 0
        last_save = time.time(); opt.zero_grad(set_to_none=True)
        done_b = batch_off; stopped = False
        for x, y in tl:
            x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            xm, ym = mixup_fn(x, y)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                loss = criterion(net(xm).float(), ym) / ACCUM
            scaler.scale(loss).backward()
            done_b += 1; run_loss += loss.item() * ACCUM; n_loss += 1
            if done_b % ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(base.parameters(), CFG.GRAD_CLIP)
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True); sched.step(); ema.update(base)
                if time.time() - last_save > CFG.SAVE_EVERY_MIN * 60:
                    ok = _save_ckpt(mid_ckpt, base, ema, opt, scaler, epoch, best_f1, done_b)
                    last_save = time.time()
                    log(f"[fold{fold}] ckpt tengah-epoch {'OK' if ok else 'GAGAL'} (ep{epoch} {done_b}/{raw_batches}, {elapsed_h():.2f}h)")
                if not time_for_one_more_interval_ok():
                    ok = _save_ckpt(mid_ckpt, base, ema, opt, scaler, epoch, best_f1, done_b)
                    log(f"[fold{fold}] STOP di tengah ep{epoch} ({done_b}/{raw_batches}): budget {CFG.TIME_BUDGET_H}h hampir habis. "
                        + ("Checkpoint tersimpan, resume otomatis." if ok else "PERINGATAN: ckpt GAGAL disimpan."))
                    stopped = True; break
        if stopped:
            return best_f1, False

        ep_h = (time.time() - t0) / 3600.0
        if batch_off == 0:
            EPOCH_TIME_H = ep_h if EPOCH_TIME_H is None else 0.5 * (EPOCH_TIME_H + ep_h)

        eval_net = nn.DataParallel(ema.module) if N_GPUS > 1 else ema.module
        vp = predict_probs(eval_net, vl)
        vy = va.target.values
        vf1 = f1_score(vy, vp.argmax(1), average="macro")
        per_cls = f1_score(vy, vp.argmax(1), average=None)
        vf1w = f1_score(vy, vp.argmax(1), average="macro", sample_weight=va.shift_w.values)   # berbobot-shift (mirip test)
        log(f"[fold{fold}] ep{epoch} loss={run_loss/max(1,n_loss):.4f} valMacroF1={vf1:.5f} shiftW={vf1w:.5f} "
            f"(per-kelas {dict(zip(CFG.CLASSES, per_cls.round(4)))}) | {ep_h*3600:.0f}s | total {elapsed_h():.2f}h")

        if vf1 > best_f1:
            best_f1, patience = vf1, 0
            _save_ckpt(best_ckpt, base, ema, opt, scaler, epoch, best_f1, light=True)
        else:
            patience += 1
        _save_ckpt(last_ckpt, base, ema, opt, scaler, epoch, best_f1)
        _cleanup(mid_ckpt)
        hp = CFG.LOGS_DIR / f"history_{CFG.MODEL_NAME}_fold{fold}.csv"
        pd.DataFrame([dict(epoch=epoch, loss=run_loss / max(1, n_loss), val_macro_f1=vf1,
                           epoch_hours=round(ep_h, 3), mixup_active=(mixup_fn is mixup_on))]
                     ).to_csv(hp, index=False, mode="a", header=not hp.exists())
        if patience >= CFG.EARLY_STOP_PATIENCE:
            log(f"[fold{fold}] early stop @ep{epoch}"); break

    _cleanup(last_ckpt); _cleanup(mid_ckpt)
    json.dump(dict(best_f1=float(best_f1), img_size=CFG.IMG_SIZE, epochs=CFG.EPOCHS), open(mdir / f"fold{fold}_done.json", "w"))
    del base, net, ema, opt; gc.collect(); torch.cuda.empty_cache()
    return best_f1, True

## 9. Jalankan Training
Fold yang sudah selesai di sesi lama dilewati. Berhenti rapi bila anggaran waktu hampir habis.

In [11]:
cv_scores, fold_done = {}, {}
for fold in CFG.TRAIN_FOLDS:
    if fold_done_info(fold) is None and not time_left_ok():
        log(f"Lewati fold{fold}: anggaran waktu hampir habis ({elapsed_h():.2f}h)."); fold_done[fold] = False; continue
    f1, done = train_one_fold(fold)
    cv_scores[fold], fold_done[fold] = f1, done
    log(f"== fold{fold}: best val MacroF1 = {f1:.5f} | selesai: {done}")
vals = [v for v in cv_scores.values() if v > 0]
if vals: log(f"==== CV Macro F1 (argmax, tanpa TTA/bias) fold {list(cv_scores)}: {np.mean(vals):.5f} ± {np.std(vals):.5f}")

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

2026-10-02 09:58:30,565 | INFO | [fold0] train=9634 val=2447 | steps/epoch=271 | EMA decay=0.9926
2026-10-02 10:08:40,617 | INFO | [fold0] ckpt tengah-epoch OK (ep0 248/543, 0.30h)
2026-10-02 10:18:52,878 | INFO | [fold0] ckpt tengah-epoch OK (ep0 494/543, 0.47h)
2026-10-02 10:22:32,344 | INFO | [fold0] ep0 loss=0.8055 valMacroF1=0.80937 shiftW=0.86304 (per-kelas {'Aman': np.float64(0.8945), 'Rusak': np.float64(0.6349), 'Lainnya': np.float64(0.8987)}) | 1343s | total 0.53h
2026-10-02 10:32:48,914 | INFO | [fold0] ckpt tengah-epoch OK (ep1 244/543, 0.70h)
2026-10-02 10:42:59,733 | INFO | [fold0] ckpt tengah-epoch OK (ep1 490/543, 0.87h)
2026-10-02 10:46:48,438 | INFO | [fold0] ep1 loss=0.5377 valMacroF1=0.88111 shiftW=0.91320 (per-kelas {'Aman': np.float64(0.9264), 'Rusak': np.float64(0.775), 'Lainnya': np.float64(0.9419)}) | 1348s | total 0.93h
2026-10-02 10:57:14,869 | INFO | [fold0] ckpt tengah-epoch OK (ep2 244/543, 1.11h)
2026-10-02 11:07:26,428 | INFO | [fold0] ckpt tengah-epoch O

2026-10-02 12:00:17,101 | INFO | [fold1] train=9654 val=2431 | steps/epoch=271 | EMA decay=0.9926
2026-10-02 12:10:22,255 | INFO | [fold1] ckpt tengah-epoch OK (ep0 244/543, 2.33h)
2026-10-02 12:20:31,358 | INFO | [fold1] ckpt tengah-epoch OK (ep0 490/543, 2.49h)
2026-10-02 12:24:19,603 | INFO | [fold1] ep0 loss=0.8035 valMacroF1=0.82703 shiftW=0.86303 (per-kelas {'Aman': np.float64(0.9035), 'Rusak': np.float64(0.6563), 'Lainnya': np.float64(0.9213)}) | 1344s | total 2.56h
2026-10-02 12:34:31,761 | INFO | [fold1] ckpt tengah-epoch OK (ep1 244/543, 2.73h)
2026-10-02 12:44:37,430 | INFO | [fold1] ckpt tengah-epoch OK (ep1 490/543, 2.90h)
2026-10-02 12:48:24,386 | INFO | [fold1] ep1 loss=0.5261 valMacroF1=0.86999 shiftW=0.89712 (per-kelas {'Aman': np.float64(0.9168), 'Rusak': np.float64(0.7507), 'Lainnya': np.float64(0.9425)}) | 1340s | total 2.96h
2026-10-02 12:58:54,349 | INFO | [fold1] ckpt tengah-epoch OK (ep2 246/543, 3.13h)
2026-10-02 13:08:59,807 | INFO | [fold1] ckpt tengah-epoch 

## 10. OOF & Probabilitas Test per Fold
Hanya fold yang **selesai** (`done.json`) yang diproses, supaya probabilitas tidak pernah berasal dari checkpoint setengah jadi. Hasil per-fold disimpan di `probs/` dan otomatis dipakai ulang di sesi berikutnya.

In [12]:
def load_best_model(fold):
    p = find_in_roots(f"models/{CFG.MODEL_NAME}/fold{fold}_best.pt")
    if p is None: return None
    ck = torch.load(p, map_location="cpu", weights_only=False)
    S = ck.get("img_size", CFG.IMG_SIZE)
    base = timm.create_model(ck.get("timm_id", CFG.TIMM_ID), pretrained=False,
                             num_classes=CFG.NUM_CLASSES, img_size=S)
    base.load_state_dict(ck["model"]); base.to(DEVICE).eval()
    net = nn.DataParallel(base) if N_GPUS > 1 else base
    dcfg = timm.data.resolve_model_data_config(base)
    _, eval_tf = make_transforms(S, dcfg["mean"], dcfg["std"])
    return base, net, eval_tf, ck

def ensure_oof(fold):
    """Return (probs, rel_list, y) untuk fold, atau None jika fold belum selesai."""
    if fold_done_info(fold) is None: return None
    pn = find_in_roots(f"probs/oof_{CFG.MODEL_NAME}_fold{fold}.npy")
    pi = find_in_roots(f"probs/oof_{CFG.MODEL_NAME}_fold{fold}_index.csv")
    va = train_df[train_df.fold == fold].reset_index(drop=True)
    if pn is not None and pi is not None:
        P = np.load(pn); idx = pd.read_csv(pi)
        if len(P) == len(va) and list(idx.rel) == list(va.rel):
            return P, list(idx.rel), idx.target.values
    m = load_best_model(fold)
    if m is None: return None
    base, net, eval_tf, ck = m
    vl = DataLoader(PkgDataset(va, eval_tf), batch_size=GLOBAL_BS * 2, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    P = predict_probs(net, vl, tta=CFG.TTA)
    np.save(CFG.PROBS_DIR / f"oof_{CFG.MODEL_NAME}_fold{fold}.npy", P)
    va[["rel", "target", "fold"]].to_csv(CFG.PROBS_DIR / f"oof_{CFG.MODEL_NAME}_fold{fold}_index.csv", index=False)
    log(f"[OOF] fold{fold} MacroF1 (TTA={CFG.TTA}) = {f1_score(va.target.values, P.argmax(1), average='macro'):.5f}")
    del base, net; gc.collect(); torch.cuda.empty_cache()
    return P, list(va.rel), va.target.values

def ensure_test_probs(fold):
    if fold_done_info(fold) is None: return None
    pn = find_in_roots(f"probs/test_{CFG.MODEL_NAME}_fold{fold}.npy")
    if pn is not None:
        P = np.load(pn)
        if len(P) == len(test_df): return P
    m = load_best_model(fold)
    if m is None: return None
    base, net, eval_tf, ck = m
    tl = DataLoader(PkgDataset(test_df, eval_tf, with_target=False), batch_size=GLOBAL_BS * 2, shuffle=False,
                    num_workers=CFG.NUM_WORKERS, pin_memory=True)
    P = predict_probs(net, tl, tta=CFG.TTA)
    np.save(CFG.PROBS_DIR / f"test_{CFG.MODEL_NAME}_fold{fold}.npy", P)
    log(f"[TEST] fold{fold}: probabilitas test tersimpan {P.shape}")
    del base, net; gc.collect(); torch.cuda.empty_cache()
    return P

test_df[["filename"]].to_csv(CFG.PROBS_DIR / "test_index.csv", index=False)
oof_parts, test_parts = {}, {}
for f in range(CFG.N_FOLDS):
    if fold_done_info(f) is None:
        log(f"[fold{f}] belum selesai -> OOF/test dilewati"); continue
    if elapsed_h() > CFG.TIME_BUDGET_H:
        log("Budget habis, sisa OOF/test dikerjakan di sesi berikutnya."); break
    r = ensure_oof(f)
    if r is not None: oof_parts[f] = r
    t = ensure_test_probs(f)
    if t is not None: test_parts[f] = t
log(f"Fold dengan OOF: {sorted(oof_parts)} | fold dengan prob test: {sorted(test_parts)}")

2026-10-02 20:08:48,793 | INFO | [OOF] fold0 MacroF1 (TTA=True) = 0.90236
2026-10-02 20:13:03,509 | INFO | [TEST] fold0: probabilitas test tersimpan (3057, 3)
2026-10-02 20:16:33,191 | INFO | [OOF] fold1 MacroF1 (TTA=True) = 0.89026
2026-10-02 20:20:48,268 | INFO | [TEST] fold1: probabilitas test tersimpan (3057, 3)
2026-10-02 20:24:16,640 | INFO | [OOF] fold2 MacroF1 (TTA=True) = 0.89340
2026-10-02 20:28:31,645 | INFO | [TEST] fold2: probabilitas test tersimpan (3057, 3)
2026-10-02 20:32:05,421 | INFO | [OOF] fold3 MacroF1 (TTA=True) = 0.89031
2026-10-02 20:36:20,833 | INFO | [TEST] fold3: probabilitas test tersimpan (3057, 3)
2026-10-02 20:39:47,230 | INFO | [OOF] fold4 MacroF1 (TTA=True) = 0.89361
2026-10-02 20:44:02,442 | INFO | [TEST] fold4: probabilitas test tersimpan (3057, 3)
2026-10-02 20:44:02,820 | INFO | Fold dengan OOF: [0, 1, 2, 3, 4] | fold dengan prob test: [0, 1, 2, 3, 4]


## 11. Evaluasi OOF + Bias Per-Kelas + Twin Blend
OOF sekarang **tanpa kebocoran kembaran** (fold berkelompok). Dua ukuran dilaporkan: biasa dan **berbobot-shift** (meniru distribusi test dari adversarial validation).
- **Bias per-kelas** dioptimasi pada Macro F1 berbobot-shift; dipakai bila gain ≥ `BIAS_MIN_GAIN`.
- **Twin blend**: untuk gambar yang punya kembaran berlabel di train, `logit[label_kembaran] += w`. Pada OOF, kembaran selalu berada di fold yang sama (tidak pernah dilihat model) sehingga bobot `w` dipilih dari simulasi yang jujur. `w = 0` bila model sudah lebih baik.

In [13]:
def macro_f1(y, pred, sw=None):
    return f1_score(y, pred, average="macro", sample_weight=sw)

def adjust_logits(P, bias, twin_label=None, w_twin=0.0):
    lg = np.log(np.clip(P, 1e-7, 1.0)) + bias
    if twin_label is not None and w_twin > 0:
        m = np.where(np.asarray(twin_label) >= 0)[0]
        lg[m, np.asarray(twin_label)[m]] += w_twin
    return lg

def optimize_bias(P, y, sw=None, grid=np.arange(-1.5, 1.501, 0.05), passes=3):
    logp = np.log(np.clip(P, 1e-7, 1.0)); K = P.shape[1]
    b = np.zeros(K); best = macro_f1(y, (logp + b).argmax(1), sw)
    order = sorted(grid, key=abs)          # tie-break: pilih offset terkecil
    for _ in range(passes):
        for c in range(K):
            cand_best, cand_val = best, b[c]
            for g in order:
                bb = b.copy(); bb[c] = g
                s = macro_f1(y, (logp + bb).argmax(1), sw)
                if s > cand_best + 1e-9: cand_best, cand_val = s, g
            b[c], best = cand_val, cand_best
    return b - b.mean(), best

BIAS, W_TWIN = np.zeros(CFG.NUM_CLASSES), 0.0
if oof_parts:
    folds_ok = sorted(oof_parts)
    P_oof = np.concatenate([oof_parts[f][0] for f in folds_ok])
    y_oof = np.concatenate([oof_parts[f][2] for f in folds_ok])
    rel2row = {r: i for i, r in enumerate(train_df.rel)}
    rows = np.array([rel2row[r] for f in folds_ok for r in oof_parts[f][1]])
    w_shift = train_df.shift_w.values[rows]
    sw_oof = w_shift if CFG.BIAS_USE_SHIFT_W else None
    tw_oof = train_df.twin_label.values[rows]
    pred0 = P_oof.argmax(1)
    log(f"OOF Macro F1 BERKELOMPOK ({len(y_oof)} sampel, fold {folds_ok}): argmax = {macro_f1(y_oof, pred0):.5f} | "
        f"berbobot-shift = {macro_f1(y_oof, pred0, w_shift):.5f}   (eva001: 0.905 bocor, public 0.83)")
    print(classification_report(y_oof, pred0, target_names=CFG.CLASSES, digits=4))
    print("Confusion matrix (baris=asli, kolom=prediksi):\n",
          pd.DataFrame(confusion_matrix(y_oof, pred0), index=CFG.CLASSES, columns=CFG.CLASSES))
    mt = tw_oof >= 0
    if mt.any():
        log(f"Subset OOF yg punya kembaran (dHash<={CFG.TWIN_HAM}): n={int(mt.sum())} | akurasi model {np.mean(pred0[mt]==y_oof[mt]):.4f} "
            f"| akurasi salin label kembaran {np.mean(tw_oof[mt]==y_oof[mt]):.4f}")
    base_obj = macro_f1(y_oof, pred0, sw_oof)
    if CFG.OPTIMIZE_BIAS:
        b, f1_b = optimize_bias(P_oof, y_oof, sw_oof)
        if f1_b - base_obj >= CFG.BIAS_MIN_GAIN:
            BIAS = b; log(f"Bias dipakai: {dict(zip(CFG.CLASSES, BIAS.round(3)))} -> Macro F1 (objektif) {base_obj:.5f} -> {f1_b:.5f}")
        else:
            log(f"Bias tidak dipakai (gain {f1_b-base_obj:+.5f} < {CFG.BIAS_MIN_GAIN})")
    if CFG.USE_TWIN_BLEND and mt.any():
        cur = best_s = macro_f1(y_oof, adjust_logits(P_oof, BIAS).argmax(1), sw_oof)
        for wt in CFG.TWIN_W_GRID:
            s = macro_f1(y_oof, adjust_logits(P_oof, BIAS, tw_oof, wt).argmax(1), sw_oof)
            if s > best_s + 1e-4: best_s, W_TWIN = s, wt
        log(f"Twin blend: w={W_TWIN} | Macro F1 (objektif) {cur:.5f} -> {best_s:.5f}")
    fin = adjust_logits(P_oof, BIAS, tw_oof, W_TWIN).argmax(1)
    log(f"OOF akhir (bias + twin): Macro F1 {macro_f1(y_oof, fin):.5f} | berbobot-shift {macro_f1(y_oof, fin, w_shift):.5f}")
else:
    log("Belum ada OOF (belum ada fold selesai) -> tanpa bias / twin.")


2026-10-02 20:44:02,947 | INFO | OOF Macro F1 BERKELOMPOK (12224 sampel, fold [0, 1, 2, 3, 4]): argmax = 0.89398 | berbobot-shift = 0.91270   (eva001: 0.905 bocor, public 0.83)
              precision    recall  f1-score   support

        Aman     0.9340    0.9336    0.9338      8800
       Rusak     0.8111    0.8141    0.8126      3001
     Lainnya     0.9446    0.9267    0.9356       423

    accuracy                         0.9040     12224
   macro avg     0.8965    0.8915    0.8940     12224
weighted avg     0.9042    0.9040    0.9041     12224

Confusion matrix (baris=asli, kolom=prediksi):
          Aman  Rusak  Lainnya
Aman     8216    567       17
Rusak     552   2443        6
Lainnya    29      2      392
2026-10-02 20:44:02,966 | INFO | Subset OOF yg punya kembaran (dHash<=4): n=3844 | akurasi model 0.8949 | akurasi salin label kembaran 0.9935
2026-10-02 20:44:04,507 | INFO | Bias dipakai: {'Aman': np.float64(0.617), 'Rusak': np.float64(-0.433), 'Lainnya': np.float64(-0.183

## 12. Submission
Rata-rata probabilitas fold (+TTA hflip) → bias OOF → twin blend → `submission.csv`. Tiga varian disimpan supaya bisa memilih 2 submission final:
`submission.csv` (bias + twin), `submission_bias_only.csv`, `submission_argmax.csv` (murni argmax). Di akhir dicetak **distribusi prediksi per ukuran gambar test** untuk memeriksa jalan pintas ukuran/kualitas.

In [14]:
def build_submission(logits, path):
    labels = [ID2LABEL[int(i)] for i in logits.argmax(1)]
    prior_lab = train_df.label.value_counts().idxmax()                  # file test corrupt -> kelas terbanyak
    for i, fp in enumerate(test_df.fpath.tolist()):
        if not isinstance(fp, str): labels[i] = prior_lab
    pm = dict(zip(test_df.filename, labels))
    if CFG.SAMPLE_SUB is not None and Path(CFG.SAMPLE_SUB).exists():
        ids = pd.read_csv(CFG.SAMPLE_SUB).filename.tolist()
    else:
        ids = test_df.filename.tolist()
    sub = pd.DataFrame(dict(filename=ids, label=[pm.get(fn, prior_lab) for fn in ids]))
    sub.to_csv(path, index=False)
    return sub

if test_parts:
    P_test = np.mean([test_parts[f] for f in sorted(test_parts)], axis=0)
    np.save(CFG.PROBS_DIR / f"test_{CFG.MODEL_NAME}_ensemble.npy", P_test)
    tw_test = test_df.twin_label.values
    sub = build_submission(adjust_logits(P_test, BIAS, tw_test, W_TWIN), CFG.SUBMISSION_PATH)
    sub.to_csv(CFG.OUTPUTS_DIR / "submission.csv", index=False)
    sub_b = build_submission(adjust_logits(P_test, BIAS), CFG.OUTPUTS_DIR / "submission_bias_only.csv")
    sub_a = build_submission(adjust_logits(P_test, np.zeros(CFG.NUM_CLASSES)), CFG.OUTPUTS_DIR / "submission_argmax.csv")
    n_tw = int((tw_test >= 0).sum())
    log(f"Submission ditulis: {CFG.SUBMISSION_PATH} | {len(sub)} baris | fold: {sorted(test_parts)} | bias={dict(zip(CFG.CLASSES, BIAS.round(3)))} "
        f"| twin w={W_TWIN} (kena {n_tw} gambar) | beda vs argmax: {int((sub.label.values != sub_a.label.values).sum())}")
    print("Distribusi prediksi test:", sub.label.value_counts(normalize=True).round(3).to_dict())
    print("Distribusi train        :", train_df.label.value_counts(normalize=True).round(3).to_dict())
    print(sub.head())
    assert sub.label.isin(CFG.CLASSES).all() and sub.filename.is_unique
    if CFG.SAMPLE_SUB is not None:
        assert set(sub.filename) == set(pd.read_csv(CFG.SAMPLE_SUB).filename), "Filename tidak cocok dgn sample_submission"

    # pemeriksaan jalan pintas: proporsi prediksi per ukuran gambar test (mis. 224x224 yang dominan Aman di train)
    _sz = test_df.w.astype(int).astype(str) + "x" + test_df.h.astype(int).astype(str)
    _top = _sz.value_counts().head(8).index
    _pred = test_df.filename.map(dict(zip(sub.filename, sub.label)))
    _tab = pd.crosstab(_sz.where(_sz.isin(_top), "lain-lain"), _pred, normalize="index").reindex(columns=CFG.CLASSES).fillna(0).round(3)
    _tab["n_test"] = _sz.where(_sz.isin(_top), "lain-lain").value_counts()
    _tab.index.name, _tab.columns.name = "ukuran test", None
    print("\nProporsi prediksi per ukuran gambar test:\n", _tab.sort_values("n_test", ascending=False))
else:
    log("Belum ada fold selesai -> submission belum dibuat. Jalankan ulang (resume) sampai minimal 1 fold selesai.")


2026-10-02 20:44:04,712 | INFO | Submission ditulis: /kaggle/working/submission.csv | 3057 baris | fold: [0, 1, 2, 3, 4] | bias={'Aman': np.float64(0.617), 'Rusak': np.float64(-0.433), 'Lainnya': np.float64(-0.183)} | twin w=3.0 (kena 968 gambar) | beda vs argmax: 145
Distribusi prediksi test: {'Aman': 0.673, 'Rusak': 0.309, 'Lainnya': 0.018}
Distribusi train        : {'Aman': 0.72, 'Rusak': 0.246, 'Lainnya': 0.035}
          filename  label
0   test_00001.jpg   Aman
1   test_00002.jpg   Aman
2   test_00003.jpg  Rusak
3   test_00004.jpg   Aman
4  test_00005.jpeg   Aman

Proporsi prediksi per ukuran gambar test:
               Aman  Rusak  Lainnya  n_test
ukuran test                               
lain-lain    0.666  0.298    0.036    1522
224x224      0.992  0.008    0.000     520
640x640      0.207  0.793    0.000     347
675x900      0.739  0.261    0.000     280
416x416      0.342  0.658    0.000     193
3000x4000    1.000  0.000    0.000      55
736x736      0.964  0.036    0.000  

## 13. Manifest & Ringkasan Artefak

In [15]:
manifest = dict(
    model=CFG.MODEL_NAME, timm_id=CFG.TIMM_ID, img_size=CFG.IMG_SIZE, classes=CFG.CLASSES,
    seed=CFG.SEED, n_folds=CFG.N_FOLDS, folds_done=[f for f in range(CFG.N_FOLDS) if fold_done_info(f) is not None],
    cv_best_val_f1={str(k): float(v) for k, v in cv_scores.items()},
    bias=dict(zip(CFG.CLASSES, map(float, BIAS))), tta=CFG.TTA, amp_dtype=str(AMP_DTYPE),
    sampler_power=CFG.SAMPLER_POWER, loss_cw_power=CFG.LOSS_CW_POWER, ema="adaptive",
    group_ham=CFG.GROUP_HAM, noise_ham=CFG.NOISE_HAM, group_w_power=CFG.GROUP_W_POWER, epoch_samples=CFG.EPOCH_SAMPLES,
    n_noisy_dropped=int(train_df.noisy.sum()), twin_w=float(W_TWIN), twin_ham=CFG.TWIN_HAM,
)
json.dump(manifest, open(CFG.OUTPUTS_DIR / "manifest.json", "w"), indent=2)
print("=" * 70)
print(f"Waktu terpakai : {elapsed_h():.2f} jam (budget {CFG.TIME_BUDGET_H})")
print(f"Fold selesai   : {manifest['folds_done']} dari {CFG.N_FOLDS}")
print(f"Bias kelas     : {manifest['bias']}")
print("=" * 70)
for root, _, files in os.walk(CFG.OUT_ROOT):
    for fn in sorted(files):
        p = Path(root) / fn
        print(f"  {p.relative_to(CFG.OUT_ROOT)}  ({p.stat().st_size/1e6:.1f} MB)")

Waktu terpakai : 10.89 jam (budget 11.3)
Fold selesai   : [0, 1, 2, 3, 4] dari 5
Bias kelas     : {'Aman': 0.6166666666666677, 'Rusak': -0.43333333333333324, 'Lainnya': -0.18333333333333438}
  diagnosis/fold_assignments_grouped.csv  (1.5 MB)
  diagnosis/sheet_Aman.png  (1.4 MB)
  diagnosis/sheet_Lainnya.png  (1.6 MB)
  diagnosis/sheet_Rusak.png  (1.2 MB)
  diagnosis/sheet_label_conflicts.png  (1.3 MB)
  diagnosis/sheet_test.png  (1.0 MB)
  diagnosis/sheet_test_twins_d0_identik.png  (0.6 MB)
  diagnosis/sheet_test_twins_d1-3.png  (0.8 MB)
  diagnosis/sheet_test_twins_d4-6.png  (0.8 MB)
  diagnosis/test_manifest.csv  (0.8 MB)
  diagnosis/train_manifest.csv  (3.5 MB)
  logs/execution.log  (0.0 MB)
  logs/history_eva02_large_fold0.csv  (0.0 MB)
  logs/history_eva02_large_fold1.csv  (0.0 MB)
  logs/history_eva02_large_fold2.csv  (0.0 MB)
  logs/history_eva02_large_fold3.csv  (0.0 MB)
  logs/history_eva02_large_fold4.csv  (0.0 MB)
  outputs/fold_assignments_grouped.csv  (2.0 MB)
  outputs/ma

---
### Catatan
1. **Baca dulu bagian 5 dan 11.** Bandingkan: OOF berkelompok (bagian 11) ↔ skor public. Jika masih selisih besar (> 0.04), shift test belum tertangani; lihat tabel per-ukuran di bagian 12 dan `sheet_test_twins_*.png`.
2. **Jangan isi `PREV_OUTPUT_DIRS` dengan output eva001** — fold dan bobot lama dilatih pada fold acak, probabilitas OOF-nya tidak sebanding.
3. **Submit bertahap** (3/hari): (a) `submission.csv`; (b) `submission_bias_only.csv` bila (a) tidak lebih baik dari yang lama; (c) `submission_argmax.csv` sebagai pembanding. Pilih 2 final dari selisih **public vs OOF berbobot-shift**, bukan public saja (public hanya 30%).
4. **Waktu**: `EPOCH_SAMPLES=8700` ≈ jumlah gambar training eva001 per fold, sehingga ±2 jam/fold seperti sebelumnya; diagnosis menambah ±5 menit.
5. **Ablasi** (bila waktu/kuota ada): matikan satu per satu — `COMPRESS_P=0`/`DOWNSCALE_P=0`, `USE_TWIN_BLEND=False`, `NOISE_HAM=-1` — dan lihat efeknya di public untuk write-up.
6. Rekomendasi write-up: jelaskan temuan diagnosis (kembaran 44%, adversarial AUC 0.87, jalan pintas metadata, label noise 95 kelompok), fold berkelompok, degradasi test-like, bobot shift, twin blend, dan EVA02-L 5-fold + TTA.